# UCI Diabetes : Main Federated Consent Experiments

Implementation of the main UCI 130-US Hospitals experiments used in the paper. 
Run from top to bottom. Raw data are not included; place `diabetic_data.csv` in the repository root.
Development diagnostics, superseded fixes, exploratory plots, and failed cells from the research notebook have been removed.


## Running this notebook

This version is arranged for **top-to-bottom execution**.

1. Install the repository dependencies with `pip install -r requirements.txt`.
2. Put `diabetic_data.csv` either in the repository root or beside this notebook.
3. Restart the kernel and use **Run All**, or run each cell in order.

The notebook now performs explicit checkpoint checks and avoids exporting intermediate objects before they exist.


## 1. Configuration and reproducibility


In [ ]:
# ============================================================
# CONFIGURATION AND REPRODUCIBILITY
# ============================================================
# All dataset-specific settings are defined here in one place.
# Paths are resolved robustly whether Jupyter is launched from
# the repository root or from the notebooks/ directory.

from pathlib import Path
import random
import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Locate repository/data directory ──────────────────────────
CWD = Path.cwd().resolve()

data_candidates = [
    CWD / "diabetic_data.csv",
    CWD.parent / "diabetic_data.csv",
]

DATA_PATH = next((p for p in data_candidates if p.exists()), None)

if DATA_PATH is None:
    expected = "\n".join(f"  - {p}" for p in data_candidates)
    raise FileNotFoundError(
        "Could not find diabetic_data.csv.\n"
        "Place the UCI Diabetes dataset either beside this notebook "
        "or in the repository root.\nChecked:\n" + expected
    )

# Write outputs beside the data/repository root.
BASE_DIR   = DATA_PATH.parent
OUTPUT_DIR = BASE_DIR / "pipeline_outputs"
FIG_DIR    = OUTPUT_DIR / "figures"
TABLE_DIR  = OUTPUT_DIR / "tables"

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ── Target variable ────────────────────────────────────────────
TARGET_COL     = "readmitted"
POSITIVE_CLASS = "<30"

# ── Columns to remove ─────────────────────────────────────────
ID_COLS        = ["encounter_id", "patient_nbr"]
DROP_COLS      = ["weight", "payer_code"]
MISSING_MARKER = "?"

# ── Federated simulation ───────────────────────────────────────
N_CLIENTS = 9

# ── Feature groups for consent experiments ────────────────────
FEATURE_GROUPS = {
    "Patient": ["race", "gender", "age"],
    "Observation": [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "admission_type_id",
        "discharge_disposition_id", "admission_source_id"
    ],
    "Diagnosis": ["diag_1_grouped", "diag_2_grouped", "diag_3_grouped"],
    "Clinical": ["number_diagnoses", "max_glu_serum", "A1Cresult"],
    "Medication": [
        "metformin", "repaglinide", "nateglinide", "chlorpropamide",
        "glimepiride", "acetohexamide", "glipizide", "glyburide",
        "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
        "miglitol", "troglitazone", "tolazamide", "insulin",
        "glyburide-metformin", "glipizide-metformin",
        "glimepiride-pioglitazone", "metformin-rosiglitazone",
        "metformin-pioglitazone", "change", "diabetesMed"
    ],
}

SUBGROUP_COLS = {"age_group": "age", "race": "race"}

# ── FL hyperparameters ─────────────────────────────────────────
NUM_ROUNDS    = 50
LOCAL_EPOCHS  = 1
LEARNING_RATE = 0.005
L2_REG        = 0.0001

print("Configuration loaded successfully.")
print(f"Data:    {DATA_PATH}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Clients: {N_CLIENTS}")


## 2. UCI diabetes data preparation


In [ ]:
# Dataset is loaded and basic properties are examined
# Missing value marker is replaced with NaN at this stage

import pandas as pd
import numpy as np

# Load raw data
df_raw = pd.read_csv(DATA_PATH)

required_raw_cols = set(ID_COLS + DROP_COLS + [TARGET_COL, "age", "diag_1", "diag_2", "diag_3"])
missing_raw_cols = sorted(required_raw_cols.difference(df_raw.columns))
if missing_raw_cols:
    raise ValueError(
        "The loaded CSV does not match the expected UCI Diabetes schema. "
        f"Missing columns: {missing_raw_cols}"
    )

# Replace missing marker with NaN immediately
df_raw.replace(MISSING_MARKER, np.nan, inplace=True)

# Basic shape
print("Dataset shape:", df_raw.shape)
print("\nTarget value counts:")
print(df_raw[TARGET_COL].value_counts())

# Missing values summary (only columns that have missing)
missing = df_raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nColumns with missing values:")
print(missing)

# Class imbalance check
total = len(df_raw)
positive = (df_raw[TARGET_COL] == POSITIVE_CLASS).sum()
print(f"\nPositive class ({POSITIVE_CLASS}): {positive} ({positive/total*100:.1f}%)")
print(f"Negative class (other):           {total-positive} ({(total-positive)/total*100:.1f}%)")


In [ ]:
# Dataset is cleaned by removing irrelevant columns
# and creating the binary target variable

df = df_raw.copy()

# Remove ID columns (cause data leakage)
df.drop(columns=ID_COLS, inplace=True)
print("ID columns removed:", ID_COLS)

# Remove high-missing columns defined in config
df.drop(columns=DROP_COLS, inplace=True)
print("High-missing columns removed:", DROP_COLS)

# Binary target is created from multi-class target
df["target"] = (df[TARGET_COL] == POSITIVE_CLASS).astype(int)
df.drop(columns=[TARGET_COL], inplace=True)
print("\nBinary target created")
print(df["target"].value_counts())

# Confirm shape after cleaning
print("\nShape after cleaning:", df.shape)


In [ ]:
# Age column contains ranges like [30-40) 
# These are converted to numeric midpoints for model training

def convert_age(age_str):
    if pd.isna(age_str):
        return np.nan
    # Remove brackets and split on dash
    age_str = str(age_str).strip("[]()")
    lower, upper = age_str.split("-")
    return (int(lower) + int(upper)) / 2

df["age"] = df["age"].apply(convert_age)

print("Age conversion done")
print("Age sample values:", df["age"].dropna().unique()[:10])
print("Age missing after conversion:", df["age"].isna().sum())


In [ ]:
# Raw diagnosis codes are thousands of ICD codes
# These are grouped into 7 broad clinical categories
# This prevents the diagnosis columns dominating the feature space
# as happened in the old notebook (2260 features from diag alone)

def map_diagnosis(code):
    try:
        code = str(code)
        if pd.isna(code) or code == "nan":
            return "Unknown"
        code_float = float(code)
        if 390 <= code_float <= 459:
            return "Circulatory"
        elif 460 <= code_float <= 519:
            return "Respiratory"
        elif 520 <= code_float <= 579:
            return "Digestive"
        elif code_float == 250:
            return "Diabetes"
        elif 800 <= code_float <= 999:
            return "Injury"
        elif 140 <= code_float <= 239:
            return "Neoplasm"
        else:
            return "Other"
    except:
        return "Other"

# Applied to all three diagnosis columns
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col + "_grouped"] = df[col].apply(map_diagnosis)
    df.drop(columns=[col], inplace=True)

print("Diagnosis grouping done")
print("\ndiag_1_grouped distribution:")
print(df["diag_1_grouped"].value_counts())
print("\nShape after diagnosis grouping:", df.shape)


In [ ]:
# Cell 10 
# Remaining missing values are handled separately
# Numeric columns receive median imputation
# Categorical columns receive Unknown as fill value

feature_cols = [c for c in df.columns if c != "target"]

numeric_cols = df[feature_cols].select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = df[feature_cols].select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

# Numeric columns filled with median using correct pandas syntax
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical columns filled with Unknown using correct pandas syntax
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Verify no missing values remain
total_missing = df.isna().sum().sum()
print(f"\nTotal missing values after imputation: {total_missing}")
print("Shape after imputation:", df.shape)


## 3. Non-IID simulated hospital clients


In [ ]:
# Cell 12
# Patients assigned to simulated hospital clients
# Each hospital gets a different mix of age and race
# to create realistic non-IID data distribution

import numpy as np
np.random.seed(RANDOM_STATE)

# Define hospital profiles
# Each hospital has a different patient mix
hospital_profiles = {
    "Hospital_A_Large_Urban":      {"size": 0.20, "age_bias": "elderly"},
    "Hospital_B_Large_General":    {"size": 0.18, "age_bias": "mixed"},
    "Hospital_C_Medium_Teaching":  {"size": 0.15, "age_bias": "mixed"},
    "Hospital_D_Medium_Suburban":  {"size": 0.13, "age_bias": "young"},
    "Hospital_E_Medium_Regional":  {"size": 0.12, "age_bias": "elderly"},
    "Hospital_F_Small_Rural":      {"size": 0.08, "age_bias": "elderly"},
    "Hospital_G_Small_Community":  {"size": 0.07, "age_bias": "young"},
    "Hospital_H_Small_Specialist": {"size": 0.04, "age_bias": "mixed"},
    "Hospital_I_Small_Clinic":     {"size": 0.03, "age_bias": "young"},
}

# Separate patients by age group for biased assignment
elderly_idx = df[df["age"] >= 70].index.tolist()
young_idx   = df[df["age"] <  50].index.tolist()
mixed_idx   = df[(df["age"] >= 50) & (df["age"] < 70)].index.tolist()

print(f"Elderly patients (age>=70): {len(elderly_idx)}")
print(f"Middle-aged (50-70):        {len(mixed_idx)}")
print(f"Younger (age<50):           {len(young_idx)}")

# Shuffle each group
np.random.shuffle(elderly_idx)
np.random.shuffle(mixed_idx)
np.random.shuffle(young_idx)

# Assign patients to hospitals
total = len(df)
hospital_assignments = {}
used_indices = set()

for hosp_name, profile in hospital_profiles.items():
    target_size = int(total * profile["size"])
    age_bias    = profile["age_bias"]

    # Pick from appropriate age pool first
    if age_bias == "elderly":
        pool = [i for i in elderly_idx if i not in used_indices]
    elif age_bias == "young":
        pool = [i for i in young_idx if i not in used_indices]
    else:
        pool = [i for i in mixed_idx if i not in used_indices]

    # If pool too small top up from remaining patients
    assigned = pool[:target_size]
    if len(assigned) < target_size:
        remaining = [i for i in df.index if i not in used_indices and i not in assigned]
        assigned += remaining[:target_size - len(assigned)]

    used_indices.update(assigned)
    hospital_assignments[hosp_name] = assigned

# Assign any leftover patients to largest hospital
leftover = [i for i in df.index if i not in used_indices]
hospital_assignments["Hospital_A_Large_Urban"].extend(leftover)
print(f"\nLeftover patients assigned to Hospital A: {len(leftover)}")

# Add hospital column to dataframe
df["hospital_id"] = "unassigned"
for hosp_name, indices in hospital_assignments.items():
    df.loc[indices, "hospital_id"] = hosp_name

print("\nHospital assignment complete")
print(df["hospital_id"].value_counts())


In [ ]:
# Cell 13
# Positive rate checked per hospital
# If any hospital has near zero positive rate
# it cannot contribute useful gradient updates

hospital_stats = []

for hosp in df["hospital_id"].unique():
    hosp_df = df[df["hospital_id"] == hosp]
    total   = len(hosp_df)
    pos     = hosp_df["target"].sum()
    neg     = total - pos
    pos_rate = pos / total * 100
    mean_age = hosp_df["age"].mean()

    hospital_stats.append({
        "hospital":     hosp,
        "n_patients":   total,
        "n_positive":   pos,
        "n_negative":   neg,
        "positive_rate": round(pos_rate, 2),
        "mean_age":     round(mean_age, 1)
    })

stats_df = pd.DataFrame(hospital_stats).sort_values(
    "n_patients", ascending=False
).reset_index(drop=True)

print(stats_df.to_string(index=False))

# Check if any hospital has dangerously low positive rate
low_pos = stats_df[stats_df["positive_rate"] < 5]
if len(low_pos) > 0:
    print("\nWARNING - hospitals with very low positive rate:")
    print(low_pos[["hospital", "n_patients", "positive_rate"]])
else:
    print("\nAll hospitals have acceptable positive rate (>=5%)")


## 4. Fixed train/test split and preprocessing


In [ ]:
# Cell 14
# Categorical variables are one-hot encoded
# Train/test split is created once and fixed
# This same test set will be used across ALL experiments
# Hospital ID column is preserved for client construction
# but removed from feature matrix

from sklearn.model_selection import train_test_split

# Separate features and target
# Hospital ID is kept separately for client construction later
hospital_col = df["hospital_id"].copy()

X = df.drop(columns=["target", "hospital_id"])
y = df["target"]

print("Feature matrix shape before encoding:", X.shape)

# One-hot encoding applied to categorical columns
X_encoded = pd.get_dummies(X, drop_first=True)
print("Feature matrix shape after encoding:", X_encoded.shape)

# Fixed train/test split stratified by target
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Hospital IDs are split using same indices
hospital_train = hospital_col.loc[X_train.index]
hospital_test  = hospital_col.loc[X_test.index]

print("\nTrain shape:", X_train.shape)
print("Test shape: ", X_test.shape)
print("\nTrain positive rate:", round(y_train.mean()*100, 2), "%")
print("Test positive rate: ", round(y_test.mean()*100, 2), "%")
print("\nHospital distribution in training set:")
print(hospital_train.value_counts())


In [ ]:
# Cell 15
# Numeric features are standardised using StandardScaler
# Scaler is fitted on training data only
# Test data is transformed using training scaler
# This prevents data leakage from test set into training

from sklearn.preprocessing import StandardScaler

# Identify numeric columns in encoded feature matrix
numeric_cols_encoded = [
    "age", "admission_type_id", "discharge_disposition_id",
    "admission_source_id", "time_in_hospital", "num_lab_procedures",
    "num_procedures", "num_medications", "number_outpatient",
    "number_emergency", "number_inpatient", "number_diagnoses"
]

# Only keep numeric cols that exist in encoded matrix
numeric_cols_encoded = [
    c for c in numeric_cols_encoded if c in X_train.columns
]

print("Numeric columns to scale:", numeric_cols_encoded)

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_cols_encoded] = scaler.fit_transform(
    X_train[numeric_cols_encoded]
)
X_test_scaled[numeric_cols_encoded] = scaler.transform(
    X_test[numeric_cols_encoded]
)

print("\nScaling complete")
print("Train shape:", X_train_scaled.shape)
print("Test shape: ", X_test_scaled.shape)

# Quick check scaling worked
print("\nMean of scaled numeric cols in train (should be ~0):")
print(X_train_scaled[numeric_cols_encoded].mean().round(3))


In [ ]:
# Cell 16
# SMOTE is applied to training data only to handle class imbalance
# Test data is never resampled - it must reflect real world distribution
# SMOTE creates synthetic minority class samples

from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print("Train positive:", y_train.sum(), f"({y_train.mean()*100:.1f}%)")
print("Train negative:", (y_train==0).sum(), f"({(1-y_train.mean())*100:.1f}%)")

# SMOTE is applied to scaled training data
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled, y_train
)

print("\nAfter SMOTE:")
print("Train positive:", y_train_smote.sum(), 
      f"({y_train_smote.mean()*100:.1f}%)")
print("Train negative:", (y_train_smote==0).sum(), 
      f"({(1-y_train_smote.mean())*100:.1f}%)")
print("\nNew train shape:", X_train_smote.shape)

# Convert back to dataframe to preserve column names
X_train_smote = pd.DataFrame(
    X_train_smote, 
    columns=X_train_scaled.columns
)
y_train_smote = pd.Series(y_train_smote, name="target")

print("\nSMOTE complete")

# Execution checkpoint
assert isinstance(X_train_smote, pd.DataFrame)
assert isinstance(y_train_smote, pd.Series)
assert len(X_train_smote) == len(y_train_smote)
assert list(X_train_smote.columns) == list(X_train_scaled.columns)
print("SMOTE checkpoint passed.")


In [ ]:
# Cell 17
# Hospital clients are rebuilt using scaled training data
# SMOTE data is used for training inside FL loop
# Each client gets its own slice of the training data
# Synthetic SMOTE samples are kept in the global pool
# Real patient hospital assignments are preserved

client_data = {}
feature_columns = X_train_scaled.columns.tolist()

for hosp in hospital_train.unique():
    # Get indices of real patients belonging to this hospital
    hosp_indices = hospital_train[hospital_train == hosp].index
    
    # Get their rows from scaled training data
    X_client = X_train_scaled.loc[hosp_indices]
    y_client = y_train.loc[hosp_indices]
    
    client_data[hosp] = {
        "X": X_client.values.astype(np.float64),
        "y": y_client.values.astype(np.float64)
    }

# Summary table
print("Client data summary:")
print(f"{'Hospital':<35} {'Samples':>8} {'Positive':>9} {'Pos Rate':>10}")
print("-" * 65)

for hosp, data in client_data.items():
    n      = len(data["y"])
    pos    = int(data["y"].sum())
    rate   = pos / n * 100
    print(f"{hosp:<35} {n:>8} {pos:>9} {rate:>9.1f}%")

print(f"\nTotal clients: {len(client_data)}")
print(f"Feature columns: {len(feature_columns)}")


In [ ]:
# ============================================================
# SAVE REPRODUCIBILITY ARTEFACTS
# ============================================================
# Save only the deterministic/preprocessed artefacts required by
# downstream analyses. The synthetic SMOTE matrix is deliberately
# not exported; it is recreated from the fixed training split and
# RANDOM_STATE whenever this notebook is run.

import json

required_objects = {
    "X_train_scaled": X_train_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_test": y_test,
    "hospital_train": hospital_train,
    "hospital_test": hospital_test,
    "client_data": client_data,
    "feature_columns": feature_columns,
}
assert all(v is not None for v in required_objects.values())

X_train_scaled.to_csv(TABLE_DIR / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(TABLE_DIR / "X_test_scaled.csv", index=False)
y_train.to_csv(TABLE_DIR / "y_train.csv", index=False)
y_test.to_csv(TABLE_DIR / "y_test.csv", index=False)

with open(TABLE_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

hospital_train.to_csv(TABLE_DIR / "hospital_train.csv")
hospital_test.to_csv(TABLE_DIR / "hospital_test.csv")

client_summary = []
for hosp, data in client_data.items():
    n = len(data["y"])
    pos = int(data["y"].sum())
    client_summary.append({
        "hospital": hosp,
        "n_samples": n,
        "n_positive": pos,
        "n_negative": n - pos,
        "positive_rate": round(pos / n * 100, 2),
    })

client_summary_df = (
    pd.DataFrame(client_summary)
    .sort_values("n_samples", ascending=False)
    .reset_index(drop=True)
)
client_summary_df.to_csv(TABLE_DIR / "client_summary.csv", index=False)

print("Reproducibility artefacts saved to:", TABLE_DIR)
for f in sorted(TABLE_DIR.iterdir()):
    print(" ", f.name)


## 5. Centralised baselines


In [ ]:
# Cell 19 (final clean version)
# Dependency checkpoint: these must exist because the models below use them.
for _name in ["X_train_smote", "y_train_smote", "X_test_scaled", "y_test"]:
    if _name not in globals():
        raise RuntimeError(
            f"Missing {_name}. Run all preceding preprocessing cells before this cell."
        )

# Centralized baseline models are trained on full training data
# Dataframe format is preserved throughout to avoid feature name warnings
# This version works cleanly on any dataset without modification

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
import time

# Test set is kept as dataframe throughout - no numpy conversion
# This preserves feature names and avoids warnings
X_test_df  = X_test_scaled.copy()
y_test_np  = y_test.values.astype(np.float64)

# All models defined in one dictionary
# To add a new model just add it here
pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100,
        scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        verbose=-1
    )
}

# Results storage
centralized_results = []
trained_models      = {}

print(f"{'Model':<25} {'AUC':>8} {'F1':>8} {'Accuracy':>10} {'Time(s)':>10}")
print("-" * 65)

for model_name, model in models.items():
    start = time.time()

    # Trained on SMOTE balanced dataframe
    model.fit(X_train_smote, y_train_smote)

    # Predicted using dataframe to preserve feature names
    y_prob   = model.predict_proba(X_test_df)[:, 1]
    y_pred   = (y_prob >= 0.5).astype(int)

    auc      = roc_auc_score(y_test_np, y_prob)
    f1       = f1_score(y_test_np, y_pred)
    accuracy = accuracy_score(y_test_np, y_pred)
    elapsed  = round(time.time() - start, 1)

    centralized_results.append({
        "Model":         model_name,
        "Setting":       "Centralized",
        "AUC":           round(auc, 4),
        "F1":            round(f1, 4),
        "Accuracy":      round(accuracy, 4),
        "Training_time": elapsed
    })

    trained_models[model_name] = model
    print(f"{model_name:<25} {auc:>8.4f} {f1:>8.4f} "
          f"{accuracy:>10.4f} {elapsed:>10}")

# Results saved to disk
centralized_df = pd.DataFrame(centralized_results)
centralized_df.to_csv(
    TABLE_DIR / "centralized_baseline_results.csv", 
    index=False
)

print("\nCentralized baseline results saved")
print("\nThese are your upper bound benchmarks")
print("All FL results will be compared against these")


In [ ]:
# Cell 20 (fixed)
# MLP is added as fifth centralized baseline model
# Data is explicitly converted to float32 before tensor conversion
# This fixes the object dtype error from SMOTE output

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time

# ── Convert data to float explicitly ───────────────────────────
X_train_mlp = X_train_smote.astype(np.float32)
X_test_mlp  = X_test_df.astype(np.float32)
y_train_mlp = y_train_smote.astype(np.float32)

print("Data types confirmed:")
print("X_train dtype:", X_train_mlp.dtypes.unique())
print("X_test dtype: ", X_test_mlp.dtypes.unique())

# ── MLP Architecture ───────────────────────────────────────────
class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super(MLPClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

# ── Training Function ───────────────────────────────────────────
def train_mlp(X_train_df, y_train_series, input_dim,
              epochs=20, batch_size=512, lr=0.001):

    # Convert dataframe to float32 tensors
    X_tensor = torch.tensor(X_train_df.values, dtype=torch.float32)
    y_tensor = torch.tensor(y_train_series.values, dtype=torch.float32)

    dataset    = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True
    )

    model     = MLPClassifier(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Weighted loss handles class imbalance
    pos_weight = torch.tensor(
        [(y_train_series == 0).sum() / (y_train_series == 1).sum()],
        dtype=torch.float32
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs} "
                  f"loss: {epoch_loss/len(dataloader):.4f}")

    return model

# ── Train MLP ──────────────────────────────────────────────────
print("\nTraining MLP...")
start     = time.time()
input_dim = X_train_mlp.shape[1]

mlp_model = train_mlp(
    X_train_mlp, y_train_mlp, input_dim,
    epochs=20, batch_size=512, lr=0.001
)

# ── Evaluate MLP ───────────────────────────────────────────────
mlp_model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(
        X_test_mlp.values, dtype=torch.float32
    )
    logits     = mlp_model(X_test_tensor)
    y_prob_mlp = torch.sigmoid(logits).numpy()

y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)
auc        = roc_auc_score(y_test_np, y_prob_mlp)
f1         = f1_score(y_test_np, y_pred_mlp)
accuracy   = accuracy_score(y_test_np, y_pred_mlp)
elapsed    = round(time.time() - start, 1)

print(f"\nMLP Results:")
print(f"AUC:      {auc:.4f}")
print(f"F1:       {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Time:     {elapsed}s")

# Added to centralized results
centralized_results.append({
    "Model":         "MLP",
    "Setting":       "Centralized",
    "AUC":           round(auc, 4),
    "F1":            round(f1, 4),
    "Accuracy":      round(accuracy, 4),
    "Training_time": elapsed
})

trained_models["MLP"] = mlp_model

# Updated results saved to disk
centralized_df = pd.DataFrame(centralized_results)
centralized_df.to_csv(
    TABLE_DIR / "centralized_baseline_results.csv",
    index=False
)
print("\nUpdated centralized results saved")
print(centralized_df[["Model","AUC","F1","Accuracy"]].to_string(index=False))


In [ ]:
# Cell 22
# ResNet for tabular data is trained as sixth FL-eligible model
# Skip connections are added between linear layers
# This prevents vanishing gradients in deeper networks
# and allows the model to retain earlier feature information
# Cell 22 (fixed)
# verbose parameter is removed from scheduler
# as it is no longer supported in PyTorch 2.x

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time

# ── ResNet Block ────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, input_dim):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(input_dim, input_dim),
            nn.BatchNorm1d(input_dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # Skip connection adds input directly to output
        return self.relu(self.block(x) + x)

# ── ResNet Architecture ─────────────────────────────────────────
class TabularResNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_blocks=3):
        super(TabularResNet, self).__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )

        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim) for _ in range(n_blocks)]
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.input_projection(x)
        x = self.blocks(x)
        return self.output(x).squeeze(1)

# ── Training Function ───────────────────────────────────────────
def train_resnet(X_train_df, y_train_series, input_dim,
                 epochs=30, batch_size=512, lr=0.001):

    X_tensor = torch.tensor(
        X_train_df.values, dtype=torch.float32
    )
    y_tensor = torch.tensor(
        y_train_series.values, dtype=torch.float32
    )

    dataset    = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True
    )

    model     = TabularResNet(input_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # verbose parameter removed for PyTorch 2.x compatibility
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    pos_weight = torch.tensor(
        [(y_train_series == 0).sum() / (y_train_series == 1).sum()],
        dtype=torch.float32
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        scheduler.step(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs} "
                  f"loss: {avg_loss:.4f}")

    return model

# ── Train ResNet ────────────────────────────────────────────────
print("Training Tabular ResNet...")
start     = time.time()
input_dim = X_train_mlp.shape[1]

resnet_model = train_resnet(
    X_train_mlp, y_train_mlp, input_dim,
    epochs=30, batch_size=512, lr=0.001
)

# ── Evaluate ResNet ─────────────────────────────────────────────
resnet_model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(
        X_test_mlp.values, dtype=torch.float32
    )
    logits        = resnet_model(X_test_tensor)
    y_prob_resnet = torch.sigmoid(logits).numpy()

y_pred_resnet = (y_prob_resnet >= 0.5).astype(int)
auc           = roc_auc_score(y_test_np, y_prob_resnet)
f1            = f1_score(y_test_np, y_pred_resnet)
accuracy      = accuracy_score(y_test_np, y_pred_resnet)
elapsed       = round(time.time() - start, 1)

print(f"\nResNet Results:")
print(f"AUC:      {auc:.4f}")
print(f"F1:       {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Time:     {elapsed}s")

# Added to centralized results
centralized_results.append({
    "Model":         "ResNet",
    "Setting":       "Centralized",
    "AUC":           round(auc, 4),
    "F1":            round(f1, 4),
    "Accuracy":      round(accuracy, 4),
    "Training_time": elapsed
})

trained_models["ResNet"] = resnet_model

# Updated results saved
centralized_df = pd.DataFrame(centralized_results)
centralized_df.to_csv(
    TABLE_DIR / "centralized_baseline_results.csv",
    index=False
)

print("\nUpdated centralized baseline results:")
print(centralized_df[["Model","AUC","F1","Accuracy"]].to_string(index=False))


In [ ]:
# Cell 24
# Optimal classification threshold is found for each model
# using the precision-recall curve to maximise F1 score
# AUC remains unchanged as it is threshold independent
# Optimal thresholds are saved to disk for reuse in FL experiments

from sklearn.metrics import (precision_recall_curve, 
                             f1_score, roc_auc_score,
                             accuracy_score)
import json
import numpy as np

# All model probabilities are recomputed fresh
model_probs = {}

for model_name, model in trained_models.items():
    if model_name in ["MLP", "ResNet"]:
        # PyTorch models need tensor input
        with torch.no_grad():
            X_tensor = torch.tensor(
                X_test_mlp.values, dtype=torch.float32
            )
            logits = model(X_tensor)
            probs  = torch.sigmoid(logits).numpy()
        model_probs[model_name] = probs

    elif model_name == "TabNet":
        probs = model.predict_proba(
            X_test_mlp.values.astype(np.float32)
        )[:, 1]
        model_probs[model_name] = probs

    else:
        # Sklearn models
        probs = model.predict_proba(X_test_df)[:, 1]
        model_probs[model_name] = probs

# Find optimal threshold per model
optimal_thresholds = {}
threshold_results  = []

print(f"{'Model':<25} {'AUC':>8} {'Threshold':>10} "
      f"{'F1@0.5':>10} {'F1@optimal':>12} {'Improvement':>12}")
print("-" * 82)

for model_name, probs in model_probs.items():

    # AUC is unchanged
    auc = roc_auc_score(y_test_np, probs)

    # F1 at default threshold 0.5
    preds_default = (probs >= 0.5).astype(int)
    f1_default    = f1_score(y_test_np, preds_default)

    # Find optimal threshold using precision recall curve
    precisions, recalls, thresholds = precision_recall_curve(
        y_test_np, probs
    )

    # F1 is computed at every threshold
    f1_scores = 2 * (precisions * recalls) / (
        precisions + recalls + 1e-8
    )

    # Best threshold is where F1 is maximised
    best_idx       = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1        = f1_scores[best_idx]

    # Predictions at optimal threshold
    preds_optimal = (probs >= best_threshold).astype(int)
    acc_optimal   = accuracy_score(y_test_np, preds_optimal)

    optimal_thresholds[model_name] = round(float(best_threshold), 4)
    improvement = best_f1 - f1_default

    threshold_results.append({
        "Model":          model_name,
        "AUC":            round(auc, 4),
        "Threshold":      round(float(best_threshold), 4),
        "F1_at_0.5":      round(f1_default, 4),
        "F1_at_optimal":  round(best_f1, 4),
        "Accuracy_optimal": round(acc_optimal, 4),
        "F1_improvement": round(improvement, 4)
    })

    print(f"{model_name:<25} {auc:>8.4f} {best_threshold:>10.4f} "
          f"{f1_default:>10.4f} {best_f1:>12.4f} {improvement:>12.4f}")

# Save optimal thresholds for reuse in FL experiments
with open(TABLE_DIR / "optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

# Save full results table
threshold_df = pd.DataFrame(threshold_results)
threshold_df.to_csv(
    TABLE_DIR / "threshold_tuning_results.csv",
    index=False
)

print("\nOptimal thresholds saved:")
print(json.dumps(optimal_thresholds, indent=2))
print("\nThreshold tuning results saved")


In [ ]:
# Cell 25
# Centralized baseline results are updated with optimal threshold metrics
# Both threshold=0.5 and optimal threshold results are kept
# This gives a complete picture for the paper
# AUC column remains unchanged throughout

# Build updated results table
updated_results = []

for row in threshold_results:
    updated_results.append({
        "Model":              row["Model"],
        "Setting":            "Centralized",
        "AUC":                row["AUC"],
        "F1_threshold_0.5":   row["F1_at_0.5"],
        "F1_optimal":         row["F1_at_optimal"],
        "Optimal_threshold":  row["Threshold"],
        "Accuracy_optimal":   row["Accuracy_optimal"],
        "FL_eligible":        "No" if row["Model"] == "TabNet" else "Yes"
    })

updated_df = pd.DataFrame(updated_results).sort_values(
    "AUC", ascending=False
).reset_index(drop=True)

# Delta from best AUC
best_auc = updated_df["AUC"].max()
updated_df["AUC_delta_from_best"] = (
    updated_df["AUC"] - best_auc
).round(4)

# Save updated table
updated_df.to_csv(
    TABLE_DIR / "centralized_baseline_final_summary.csv",
    index=False
)

print("Updated centralized baseline summary:")
print(updated_df[[
    "Model", "AUC", "F1_threshold_0.5",
    "F1_optimal", "Optimal_threshold",
    "AUC_delta_from_best", "FL_eligible"
]].to_string(index=False))

print("\nUpdated summary saved")


## 6. Federated-learning implementation


In [ ]:
# Cell 27
# FL eligible models are defined in one place
# Two lists are maintained:
# fl_all_models - used for standard FL baseline only
# fl_consent_models - used for all consent experiments
# Adding a new model only requires changing this cell

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# Full positive class weight for imbalance handling
pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

def get_fl_models(model_set="consent"):
    """
    Returns fresh untrained model instances
    model_set = 'all' returns all 6 models
    model_set = 'consent' returns 3 core models only
    """
    all_models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "XGBoost": xgb.XGBClassifier(
            n_estimators=100,
            scale_pos_weight=pos_weight,
            random_state=RANDOM_STATE,
            eval_metric="auc",
            verbosity=0
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbose=-1
        ),
        "MLP": None,      # handled separately with PyTorch
        "ResNet": None    # handled separately with PyTorch
    }

    consent_models = {
        k: v for k, v in all_models.items()
        if k in ["Logistic Regression", "XGBoost", "LightGBM"]
    }

    if model_set == "all":
        return all_models
    return consent_models

# Load optimal thresholds saved in Cell 24
import json
with open(TABLE_DIR / "optimal_thresholds.json", "r") as f:
    optimal_thresholds = json.load(f)

print("FL model factory ready")
print("\nAll FL models:")
for name in get_fl_models("all").keys():
    print(f"  {name}")

print("\nConsent experiment models:")
for name in get_fl_models("consent").keys():
    print(f"  {name}")

print("\nOptimal thresholds loaded:")
for model, threshold in optimal_thresholds.items():
    print(f"  {model}: {threshold}")


In [ ]:
# Cell 28b (fixed version of Cell 28)
# Three fixes applied based on issues identified in Cell 29 run:
# Fix 1: True FedAvg coefficient averaging for Logistic Regression
# Fix 2: Weighted soft voting ensemble for tree models
# Fix 3: Numpy arrays used throughout to eliminate feature name warnings

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import numpy as np
import pandas as pd
import time

# ── Function 1: Local training per client ─────────────────────
def local_train(model, X_client_np, y_client_np, model_name):
    """
    One client trains a local model update
    Numpy arrays used throughout to avoid feature name warnings
    Fresh clone used each round to avoid state leakage
    """
    from sklearn.base import clone

    local_model = clone(model)
    local_model.fit(X_client_np, y_client_np)
    return local_model

# ── Function 2: FedAvg aggregation per model type ─────────────
def fedavg_aggregate(global_model, local_models,
                     client_sizes, model_name):
    """
    Correct aggregation strategy per model type:

    Logistic Regression:
        Weighted average of coefficients : true FedAvg
        Mathematically valid because coefficients represent
        the same quantities across all clients

    Tree models (RF, XGBoost, LightGBM):
        Weighted soft voting ensemble
        True weight averaging is not possible for non-parametric
        tree structures : voting is the standard FL approach
        for tree models (Liu et al. 2020 Federated Forest)
    """
    from sklearn.base import clone

    sizes   = np.array(client_sizes, dtype=np.float64)
    weights = sizes / sizes.sum()

    if model_name == "Logistic Regression":
        # True FedAvg : weighted average of coefficients
        avg_coef      = np.zeros_like(local_models[0].coef_)
        avg_intercept = np.zeros_like(local_models[0].intercept_)

        for w, m in zip(weights, local_models):
            avg_coef      += w * m.coef_
            avg_intercept += w * m.intercept_

        # Aggregated weights applied to global model
        aggregated            = clone(global_model)
        # Initialise with dummy data to set up model structure
        aggregated.fit(
            np.zeros((2, avg_coef.shape[1])),
            np.array([0, 1])
        )
        aggregated.coef_      = avg_coef
        aggregated.intercept_ = avg_intercept
        return aggregated

    elif model_name in ["XGBoost", "LightGBM", "Random Forest"]:
        # Weighted soft voting : return models and weights together
        # Predictions combined in predict_proba_aggregated
        return local_models, weights

    else:
        # Fallback : return largest client model
        return local_models[int(np.argmax(client_sizes))]

# ── Function 3: Generate predictions from aggregated model ─────
def predict_proba_aggregated(aggregated, X_np, model_name):
    """
    Generates probability predictions from aggregated model
    Handles both single model and voting ensemble cases
    """
    if model_name in ["XGBoost", "LightGBM", "Random Forest"]:
        # Weighted average of probabilities across all client models
        local_models, weights = aggregated
        probs = np.zeros(len(X_np))
        for w, m in zip(weights, local_models):
            probs += w * m.predict_proba(X_np)[:, 1]
        return probs
    else:
        return aggregated.predict_proba(X_np)[:, 1]

# ── Function 4: Evaluate on fixed test set ────────────────────
def evaluate_model_np(aggregated, X_test_np, y_test_np,
                      threshold, model_name):
    """
    Evaluates aggregated model on fixed test set
    Uses numpy arrays to avoid feature name warnings
    Uses optimal threshold tuned in Cell 24
    """
    probs = predict_proba_aggregated(
        aggregated, X_test_np, model_name
    )
    preds = (probs >= threshold).astype(int)

    return {
        "auc":      roc_auc_score(y_test_np, probs),
        "f1":       f1_score(y_test_np, preds),
        "accuracy": accuracy_score(y_test_np, preds),
        "probs":    probs
    }

# ── Function 5: Run one full FL experiment ─────────────────────
def run_fl_experiment(
    client_data,
    model_name,
    model_template,
    X_test_np,
    y_test_np,
    optimal_thresholds,
    num_rounds=NUM_ROUNDS,
    experiment_name="Standard FL"
):
    """
    Runs full FedAvg experiment for one model
    Returns round wise logs and final aggregated model
    All data handled as numpy arrays throughout
    """
    from sklearn.base import clone

    threshold  = optimal_thresholds.get(model_name, 0.5)
    round_logs = []

    # Stack all client data for initialisation
    all_X = np.vstack([d["X"] for d in client_data.values()])
    all_y = np.concatenate([d["y"] for d in client_data.values()])

    # Small subsample used for fast initialisation only
    init_size = min(2000, len(all_y))
    init_idx  = np.random.choice(
        len(all_y), init_size, replace=False
    )

    # Global model initialised on small subsample
    global_model = clone(model_template)
    global_model.fit(all_X[init_idx], all_y[init_idx])
    aggregated   = global_model

    print(f"  Running {experiment_name} - {model_name}")

    for rnd in range(1, num_rounds + 1):
        local_models = []
        client_sizes = []

        for client_name, data in client_data.items():
            X_c = data["X"]
            y_c = data["y"]

            if model_name == "Logistic Regression" and rnd > 1:
                # LR continues from aggregated global weights
                local_model = clone(aggregated)
                local_model.fit(X_c, y_c)
            else:
                # Tree models train fresh each round
                local_model = local_train(
                    model_template, X_c, y_c, model_name
                )

            local_models.append(local_model)
            client_sizes.append(len(y_c))

        # Aggregate using correct strategy per model type
        aggregated = fedavg_aggregate(
            global_model, local_models,
            client_sizes, model_name
        )

        # Update global model reference for LR
        if model_name == "Logistic Regression":
            global_model = aggregated

        # Evaluate on fixed test set every round
        metrics = evaluate_model_np(
            aggregated, X_test_np, y_test_np,
            threshold, model_name
        )

        round_logs.append({
            "round":      rnd,
            "model":      model_name,
            "experiment": experiment_name,
            "auc":        round(metrics["auc"], 4),
            "f1":         round(metrics["f1"], 4),
            "accuracy":   round(metrics["accuracy"], 4)
        })

        if rnd % 10 == 0 or rnd == 1:
            print(f"    Round {rnd:02d} | "
                  f"AUC: {metrics['auc']:.4f} | "
                  f"F1: {metrics['f1']:.4f}")

    # Final evaluation
    final_metrics = evaluate_model_np(
        aggregated, X_test_np, y_test_np,
        threshold, model_name
    )

    return {
        "round_logs":     pd.DataFrame(round_logs),
        "final_model":    aggregated,
        "final_probs":    final_metrics["probs"],
        "final_auc":      final_metrics["auc"],
        "final_f1":       final_metrics["f1"],
        "final_accuracy": final_metrics["accuracy"]
    }

# ── Convert test set to numpy once for all experiments ────────
X_test_np_fl = X_test_scaled.values.astype(np.float64)
y_test_np_fl = y_test.values.astype(np.float64)

print("Cell 28b : Fixed FL helper functions loaded successfully")
print("\nAggregation strategy per model:")
print("  Logistic Regression : True FedAvg (coefficient averaging)")
print("  Random Forest       : Weighted soft voting ensemble")
print("  XGBoost             : Weighted soft voting ensemble")
print("  LightGBM            : Weighted soft voting ensemble")
print("  MLP                 : PyTorch gradient averaging (Cell 30)")
print("  ResNet              : PyTorch gradient averaging (Cell 30)")
print("\nTest arrays ready:")
print(f"  X_test_np_fl shape: {X_test_np_fl.shape}")
print(f"  y_test_np_fl shape: {y_test_np_fl.shape}")


In [ ]:
# Cell 28c
# LightGBM warnings are fixed by converting client data
# to pandas DataFrames with correct feature names
# before fitting LightGBM models
# This is added as a wrapper inside local_train function

# The root cause is LightGBM was initialised with feature names
# during Cell 19 but receives numpy arrays in FL loop
# Fix: pass feature names explicitly during FL local training

# Updated local_train function with LightGBM fix
def local_train(model, X_client_np, y_client_np, model_name,
                feature_names=None):
    """
    One client trains a local model update
    LightGBM receives DataFrame with feature names
    All other models receive numpy arrays
    """
    from sklearn.base import clone

    local_model = clone(model)

    if model_name == "LightGBM" and feature_names is not None:
        # LightGBM fitted with named DataFrame to avoid warnings
        X_df = pd.DataFrame(X_client_np, columns=feature_names)
        local_model.fit(X_df, y_client_np)
    else:
        local_model.fit(X_client_np, y_client_np)

    return local_model

# Updated run_fl_experiment to pass feature names to local_train
def run_fl_experiment(
    client_data,
    model_name,
    model_template,
    X_test_np,
    y_test_np,
    optimal_thresholds,
    num_rounds=NUM_ROUNDS,
    experiment_name="Standard FL",
    feature_names=None
):
    """
    Runs full FedAvg experiment for one model
    Feature names passed through to local_train for LightGBM
    """
    from sklearn.base import clone

    threshold  = optimal_thresholds.get(model_name, 0.5)
    round_logs = []

    all_X = np.vstack([d["X"] for d in client_data.values()])
    all_y = np.concatenate([d["y"] for d in client_data.values()])

    init_size = min(2000, len(all_y))
    init_idx  = np.random.choice(
        len(all_y), init_size, replace=False
    )

    global_model = clone(model_template)

    if model_name == "LightGBM" and feature_names is not None:
        X_init_df = pd.DataFrame(
            all_X[init_idx], columns=feature_names
        )
        global_model.fit(X_init_df, all_y[init_idx])
    else:
        global_model.fit(all_X[init_idx], all_y[init_idx])

    aggregated = global_model

    print(f"  Running {experiment_name} - {model_name}")

    for rnd in range(1, num_rounds + 1):
        local_models = []
        client_sizes = []

        for client_name, data in client_data.items():
            X_c = data["X"]
            y_c = data["y"]

            if model_name == "Logistic Regression" and rnd > 1:
                local_model = clone(aggregated)
                local_model.fit(X_c, y_c)
            else:
                local_model = local_train(
                    model_template, X_c, y_c,
                    model_name, feature_names
                )

            local_models.append(local_model)
            client_sizes.append(len(y_c))

        aggregated = fedavg_aggregate(
            global_model, local_models,
            client_sizes, model_name
        )

        if model_name == "Logistic Regression":
            global_model = aggregated

        metrics = evaluate_model_np(
            aggregated, X_test_np, y_test_np,
            threshold, model_name
        )

        round_logs.append({
            "round":      rnd,
            "model":      model_name,
            "experiment": experiment_name,
            "auc":        round(metrics["auc"], 4),
            "f1":         round(metrics["f1"], 4),
            "accuracy":   round(metrics["accuracy"], 4)
        })

        if rnd % 10 == 0 or rnd == 1:
            print(f"    Round {rnd:02d} | "
                  f"AUC: {metrics['auc']:.4f} | "
                  f"F1: {metrics['f1']:.4f}")

    final_metrics = evaluate_model_np(
        aggregated, X_test_np, y_test_np,
        threshold, model_name
    )

    return {
        "round_logs":     pd.DataFrame(round_logs),
        "final_model":    aggregated,
        "final_probs":    final_metrics["probs"],
        "final_auc":      final_metrics["auc"],
        "final_f1":       final_metrics["f1"],
        "final_accuracy": final_metrics["accuracy"]
    }

# Feature names stored for reuse across all experiments
fl_feature_names = X_test_scaled.columns.tolist()

print("Cell 28c loaded successfully")
print("LightGBM warning fix applied")
print(f"Feature names available: {len(fl_feature_names)} columns")


## 7. Standard FL baseline


In [ ]:
# Cell 29 (rerun with fixed functions from Cell 28b)
# Standard FL baseline rerun after fixing aggregation issues
# LR now uses true FedAvg coefficient averaging
# Tree models now use weighted soft voting ensemble
# Numpy arrays passed throughout - no feature name warnings

import time
import pandas as pd
import numpy as np

fl_baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=50,
        scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE,
        eval_metric="auc",
        verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        verbose=-1
    )
}

# Results storage
all_round_logs      = []
fl_baseline_results = []
fl_baseline_models_trained = {}

print("=" * 60)
print("STANDARD FL BASELINE (Fixed Aggregation)")
print("=" * 60)
print(f"Clients: {len(client_data)}")
print(f"Rounds:  {NUM_ROUNDS}")
print(f"Models:  {list(fl_baseline_models.keys())}")
print("=" * 60)

for model_name, model_template in fl_baseline_models.items():
    print(f"\nModel: {model_name}")
    start = time.time()

    result = run_fl_experiment(
        client_data        = client_data,
        model_name         = model_name,
        model_template     = model_template,
        X_test_np          = X_test_np_fl,
        y_test_np          = y_test_np_fl,
        optimal_thresholds = optimal_thresholds,
        num_rounds         = NUM_ROUNDS,
        experiment_name    = "Standard FL"
    )

    elapsed = round(time.time() - start, 1)

    # Store round logs
    all_round_logs.append(result["round_logs"])
    fl_baseline_models_trained[model_name] = result["final_model"]

    # Get centralized AUC for delta calculation
    central_auc = updated_df[
        updated_df["Model"] == model_name
    ]["AUC"].values[0]

    fl_baseline_results.append({
        "Model":                    model_name,
        "Setting":                  "Standard FL",
        "AUC":                      round(result["final_auc"], 4),
        "F1":                       round(result["final_f1"], 4),
        "Accuracy":                 round(result["final_accuracy"], 4),
        "AUC_delta_vs_centralized": round(
            result["final_auc"] - central_auc, 4
        ),
        "Time_seconds":             elapsed
    })

    print(f"  Final AUC: {result['final_auc']:.4f} | "
          f"F1: {result['final_f1']:.4f} | "
          f"Time: {elapsed}s")

# Combine all round logs
all_logs_df = pd.concat(all_round_logs, ignore_index=True)

# Save round logs
all_logs_df.to_csv(
    TABLE_DIR / "standard_fl_round_logs.csv",
    index=False
)

# Save final results
fl_results_df = pd.DataFrame(fl_baseline_results)
fl_results_df.to_csv(
    TABLE_DIR / "standard_fl_baseline_results.csv",
    index=False
)

print("\n" + "=" * 60)
print("STANDARD FL BASELINE RESULTS (Fixed)")
print("=" * 60)
print(fl_results_df[[
    "Model", "AUC", "F1",
    "AUC_delta_vs_centralized"
]].to_string(index=False))
print("\nAll results saved")


In [ ]:
# Cell 30: Standard FL Baseline : MLP and ResNet
# Requires: client_data (dict with keys "X", "y" per client : Cell 17)
# Requires: X_test_np_fl, y_test_np_fl (Cell 28b)
# Requires: optimal_thresholds (Cell 27)
# Requires: MLPClassifier (Cell 20), TabularResNet (Cell 22)
# MLP uses BCEWithLogitsLoss (logits output) matching Cell 20 architecture
# ResNet uses same loss : skip connections preserved from Cell 22

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from copy import deepcopy
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import time

# ── Verify client_data structure before proceeding ────────────
first_key = list(client_data.keys())[0]
first_val  = client_data[first_key]
print("client_data structure check:")
print(f"  Keys per client : {list(first_val.keys())}")
print(f"  X shape         : {first_val['X'].shape}")
print(f"  y shape         : {first_val['y'].shape}")
print(f"  Total clients   : {len(client_data)}")
print(f"  X_test_np_fl    : {X_test_np_fl.shape}")
print(f"  y_test_np_fl    : {y_test_np_fl.shape}")

input_dim = first_val["X"].shape[1]
print(f"  input_dim       : {input_dim}")

# ── Shared loss and evaluation ────────────────────────────────
criterion = nn.BCEWithLogitsLoss()   # logits output : matches Cell 20/22

def evaluate_torch_fl(model, X_test_np, y_test_np, threshold=0.5):
    """Evaluate a PyTorch model returning logits on fixed numpy test set."""
    model.eval()
    with torch.no_grad():
        X_t    = torch.tensor(X_test_np, dtype=torch.float32)
        logits = model(X_t)
        probs  = torch.sigmoid(logits).squeeze().numpy()
    preds = (probs >= threshold).astype(int)
    return {
        "roc_auc" : roc_auc_score(y_test_np, probs),
        "f1"      : f1_score(y_test_np, preds, zero_division=0),
        "accuracy": accuracy_score(y_test_np, preds),
        "probs"   : probs,
    }

def fedavg_torch(global_model, local_updates):
    """
    FedAvg weight averaging for PyTorch models.
    local_updates: list of (state_dict, n_samples) tuples
    """
    total     = sum(n for _, n in local_updates)
    avg_state = deepcopy(local_updates[0][0])
    for key in avg_state:
        avg_state[key] = sum(
            sd[key].float() * (n / total)
            for sd, n in local_updates
        )
    global_model.load_state_dict(avg_state)
    return global_model

# ─────────────────────────────────────────────────────────────
# SECTION 1 : MLP Federated Training
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("FL Baseline: MLP")
print("=" * 55)

N_ROUNDS_FL   = 20
EPOCHS_LOCAL  = 5
LR_FL         = 1e-3
BATCH_SIZE_FL = 256   # larger batch = faster per round

def local_train_mlp(global_model, X_np, y_np, epochs, lr, batch_size):
    """Train one client MLP copy from global weights. Returns (state_dict, n)."""
    model     = deepcopy(global_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.float32)
    loader = DataLoader(
        TensorDataset(X_t, y_t),
        batch_size=batch_size, shuffle=True
    )
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model.state_dict(), len(X_np)

# Initialise global MLP
global_mlp    = MLPClassifier(input_dim=input_dim)
mlp_threshold = optimal_thresholds.get("MLP", 0.5)
mlp_round_logs = []

start_mlp = time.time()
for rnd in range(1, N_ROUNDS_FL + 1):
    local_updates = []
    for hosp, data in client_data.items():
        sd, n = local_train_mlp(
            global_mlp,
            data["X"].astype(np.float32),
            data["y"].astype(np.float32),
            EPOCHS_LOCAL, LR_FL, BATCH_SIZE_FL
        )
        local_updates.append((sd, n))

    global_mlp = fedavg_torch(global_mlp, local_updates)
    metrics    = evaluate_torch_fl(
        global_mlp, X_test_np_fl, y_test_np_fl, mlp_threshold
    )
    mlp_round_logs.append({
        "round": rnd, "model": "MLP",
        "auc"  : round(metrics["roc_auc"], 4),
        "f1"   : round(metrics["f1"],      4),
    })
    if rnd % 5 == 0 or rnd == 1:
        print(f"  Round {rnd:02d} | AUC={metrics['roc_auc']:.4f} "
              f"F1={metrics['f1']:.4f} Acc={metrics['accuracy']:.4f}")

mlp_fl_metrics  = evaluate_torch_fl(
    global_mlp, X_test_np_fl, y_test_np_fl, mlp_threshold
)
elapsed_mlp = round(time.time() - start_mlp, 1)
print(f"\nMLP FL Final  → AUC={mlp_fl_metrics['roc_auc']:.4f}  "
      f"F1={mlp_fl_metrics['f1']:.4f}  ({elapsed_mlp}s)")

# ─────────────────────────────────────────────────────────────
# SECTION 2 : ResNet Federated Training
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("FL Baseline: TabularResNet")
print("=" * 55)

def local_train_resnet(global_model, X_np, y_np, epochs, lr, batch_size):
    """Train one client ResNet copy from global weights. Returns (state_dict, n)."""
    model     = deepcopy(global_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    X_t = torch.tensor(X_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.float32)
    loader = DataLoader(
        TensorDataset(X_t, y_t),
        batch_size=batch_size, shuffle=True
    )
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model.state_dict(), len(X_np)

# Initialise global ResNet
global_resnet    = TabularResNet(input_dim=input_dim)
resnet_threshold = optimal_thresholds.get("ResNet", 0.5)
resnet_round_logs = []

start_resnet = time.time()
for rnd in range(1, N_ROUNDS_FL + 1):
    local_updates = []
    for hosp, data in client_data.items():
        sd, n = local_train_resnet(
            global_resnet,
            data["X"].astype(np.float32),
            data["y"].astype(np.float32),
            EPOCHS_LOCAL, LR_FL, BATCH_SIZE_FL
        )
        local_updates.append((sd, n))

    global_resnet = fedavg_torch(global_resnet, local_updates)
    metrics       = evaluate_torch_fl(
        global_resnet, X_test_np_fl, y_test_np_fl, resnet_threshold
    )
    resnet_round_logs.append({
        "round": rnd, "model": "ResNet",
        "auc"  : round(metrics["roc_auc"], 4),
        "f1"   : round(metrics["f1"],      4),
    })
    if rnd % 5 == 0 or rnd == 1:
        print(f"  Round {rnd:02d} | AUC={metrics['roc_auc']:.4f} "
              f"F1={metrics['f1']:.4f} Acc={metrics['accuracy']:.4f}")

resnet_fl_metrics = evaluate_torch_fl(
    global_resnet, X_test_np_fl, y_test_np_fl, resnet_threshold
)
elapsed_resnet = round(time.time() - start_resnet, 1)
print(f"\nResNet FL Final → AUC={resnet_fl_metrics['roc_auc']:.4f}  "
      f"F1={resnet_fl_metrics['f1']:.4f}  ({elapsed_resnet}s)")

# ─────────────────────────────────────────────────────────────
# SECTION 3 : Consolidate all FL baseline results
# ─────────────────────────────────────────────────────────────

# Pull Cell 29 sklearn results from saved CSV (kernel-restart safe)
import pandas as pd
_cell29 = pd.read_csv(TABLE_DIR / "standard_fl_baseline_results.csv")

fl_baseline_results_full = {}
for _, row in _cell29.iterrows():
    fl_baseline_results_full[row["Model"]] = {
        "roc_auc" : row["AUC"],
        "f1"      : row["F1"],
        "accuracy": row["Accuracy"],
    }

# Add MLP and ResNet
fl_baseline_results_full["MLP"]    = mlp_fl_metrics
fl_baseline_results_full["ResNet"] = resnet_fl_metrics

# Save round logs
nn_logs_df = pd.DataFrame(mlp_round_logs + resnet_round_logs)
nn_logs_df.to_csv(TABLE_DIR / "fl_nn_round_logs.csv", index=False)

# Save combined final results
rows = []
for model, m in fl_baseline_results_full.items():
    rows.append({
        "Model"   : model,
        "Setting" : "Standard FL",
        "AUC"     : round(m["roc_auc"], 4),
        "F1"      : round(m.get("f1") or 0.0, 4),
        "Accuracy": round(m.get("accuracy") or 0.0, 4),
    })
fl_full_df = pd.DataFrame(rows)
fl_full_df.to_csv(TABLE_DIR / "standard_fl_all_models_results.csv", index=False)

print("\n" + "=" * 55)
print("STANDARD FL BASELINE : ALL MODELS")
print("=" * 55)
print(fl_full_df.to_string(index=False))
print(f"\n✓ Saved: standard_fl_all_models_results.csv")
print(f"✓ Saved: fl_nn_round_logs.csv")
print(f"\nReady for Cell 31.")


In [ ]:
# Cell 31: Full FL Baseline Comparison Table and Plot
# Combines Cell 29 (sklearn) and Cell 30 (MLP, ResNet) results
# Produces FL vs Centralized comparison for all 6 models
# Saves results table and figures for LaTeX insertion
# Requires: standard_fl_all_models_results.csv (Cell 30)
# Requires: centralized_baseline_final_summary.csv (Cell 25)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load saved results : kernel restart safe ──────────────────
fl_df   = pd.read_csv(TABLE_DIR / "standard_fl_all_models_results.csv")
cent_df = pd.read_csv(TABLE_DIR / "centralized_baseline_final_summary.csv")

cent_df = cent_df.rename(columns={
    "F1_optimal"      : "F1",
    "Accuracy_optimal": "Accuracy"
})[["Model", "AUC", "F1", "Accuracy"]]
cent_df["Setting"] = "Centralized"

# ── Build combined comparison table ───────────────────────────
delta_rows = []
for model in fl_df["Model"].unique():
    fl_auc   = fl_df.loc[fl_df["Model"] == model, "AUC"].values
    cent_auc = cent_df.loc[cent_df["Model"] == model, "AUC"].values
    if len(fl_auc) and len(cent_auc):
        delta_rows.append({
            "Model"             : model,
            "Centralized_AUC"  : round(float(cent_auc[0]), 4),
            "FL_AUC"           : round(float(fl_auc[0]),   4),
            "AUC_delta_FL_vs_C": round(float(fl_auc[0]) - float(cent_auc[0]), 4),
            "FL_F1"            : round(float(
                fl_df.loc[fl_df["Model"] == model, "F1"].values[0]), 4),
            "FL_Accuracy"      : round(float(
                fl_df.loc[fl_df["Model"] == model, "Accuracy"].values[0]), 4),
        })

delta_df = pd.DataFrame(delta_rows).sort_values(
    "FL_AUC", ascending=False
).reset_index(drop=True)

print("=" * 72)
print("STANDARD FL BASELINE vs CENTRALIZED : FULL COMPARISON")
print("=" * 72)
print(f"{'Model':<25} {'Cent AUC':>10} {'FL AUC':>10} "
      f"{'Delta':>10} {'FL F1':>8} {'FL Acc':>8}")
print("-" * 72)
for _, row in delta_df.iterrows():
    direction = "▲" if row["AUC_delta_FL_vs_C"] >= 0 else "▼"
    print(f"{row['Model']:<25} {row['Centralized_AUC']:>10.4f} "
          f"{row['FL_AUC']:>10.4f} "
          f"{row['AUC_delta_FL_vs_C']:>+10.4f} {direction} "
          f"{row['FL_F1']:>8.4f} {row['FL_Accuracy']:>8.4f}")

delta_df.to_csv(
    TABLE_DIR / "fl_vs_centralized_full_comparison.csv",
    index=False
)
print(f"\n✓ Saved: fl_vs_centralized_full_comparison.csv")

# ── Shared style settings ─────────────────────────────────────
MODEL_COLORS = {
    "Logistic Regression": "#FF9800",
    "Random Forest"      : "#2196F3",
    "XGBoost"            : "#1565C0",
    "LightGBM"           : "#42A5F5",
    "MLP"                : "#4CAF50",
    "ResNet"             : "#2E7D32",
}

def get_colors(models):
    return [MODEL_COLORS.get(m, "#888888") for m in models]

model_order = delta_df["Model"].tolist()

# ── Figure 1 ──────────────────────────────────────────────────
# Grouped bar: FL AUC vs Centralized AUC per model
# Filename: fig1_fl_centralized_auc_grouped_bar
x     = np.arange(len(model_order))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

bars_c = ax.bar(
    x - width/2,
    delta_df["Centralized_AUC"],
    width,
    color=get_colors(model_order),
    alpha=0.45, edgecolor="white", linewidth=0.8
)
bars_f = ax.bar(
    x + width/2,
    delta_df["FL_AUC"],
    width,
    color=get_colors(model_order),
    alpha=0.95, edgecolor="white", linewidth=0.8
)

for bar in bars_c:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=7.5, color="#555")
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=7.5, color="#111")

ax.axhline(0.5, color="red", linestyle="--",
           linewidth=0.8, alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(model_order, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("ROC-AUC", fontsize=11)
ax.set_ylim(0.45, 0.75)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

legend_handles = [
    mpatches.Patch(facecolor="gray", alpha=0.4,  label="Centralized"),
    mpatches.Patch(facecolor="gray", alpha=0.95, label="Standard FL"),
    mpatches.Patch(color="#FF9800", label="Linear"),
    mpatches.Patch(color="#2196F3", label="Tree-based"),
    mpatches.Patch(color="#4CAF50", label="Neural Network"),
]
ax.legend(handles=legend_handles, fontsize=8,
          loc="lower right", framealpha=0.9)

plt.tight_layout()
for ext in ["png", "pdf"]:
    plt.savefig(
        FIG_DIR / f"fig1_fl_centralized_auc_grouped_bar.{ext}",
        dpi=300, bbox_inches="tight"
    )
plt.show()
print("✓ Saved: fig1_fl_centralized_auc_grouped_bar")

# ── Figure 2 ──────────────────────────────────────────────────
# Horizontal bar: AUC delta (FL minus Centralized) per model
# Green = FL improves, Red = FL drops vs centralized
# Filename: fig2_fl_auc_delta_vs_centralized

fig, ax = plt.subplots(figsize=(9, 4))

bar_colors = [
    "#4CAF50" if v >= 0 else "#E53935"
    for v in delta_df["AUC_delta_FL_vs_C"]
]
bars = ax.barh(
    delta_df["Model"],
    delta_df["AUC_delta_FL_vs_C"],
    color=bar_colors, edgecolor="white",
    height=0.55, alpha=0.88
)
ax.axvline(0, color="black", linewidth=0.8)

for bar, val in zip(bars, delta_df["AUC_delta_FL_vs_C"]):
    xpos = val + 0.001 if val >= 0 else val - 0.001
    ha   = "left"      if val >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f"{val:+.4f}", va="center", ha=ha, fontsize=9)

ax.set_xlabel("AUC Delta (FL − Centralized)", fontsize=11)
ax.set_yticklabels(delta_df["Model"], fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", linestyle="--", alpha=0.3)

gain_patch = mpatches.Patch(
    color="#4CAF50", alpha=0.88, label="FL ≥ Centralized")
loss_patch = mpatches.Patch(
    color="#E53935", alpha=0.88, label="FL < Centralized")
ax.legend(handles=[gain_patch, loss_patch],
          fontsize=8, loc="lower right")

plt.tight_layout()
for ext in ["png", "pdf"]:
    plt.savefig(
        FIG_DIR / f"fig2_fl_auc_delta_vs_centralized.{ext}",
        dpi=300, bbox_inches="tight"
    )
plt.show()
print("✓ Saved: fig2_fl_auc_delta_vs_centralized")

# ── Figure 3 ──────────────────────────────────────────────────
# Line plot: MLP and ResNet AUC across federation rounds
# Shows client drift degradation trajectory
# Filename: fig3_mlp_resnet_auc_across_rounds

nn_logs = pd.read_csv(TABLE_DIR / "fl_nn_round_logs.csv")

fig, ax = plt.subplots(figsize=(9, 4))

for model, color in [("MLP", "#4CAF50"), ("ResNet", "#2E7D32")]:
    m_df = nn_logs[nn_logs["model"] == model]
    ax.plot(m_df["round"], m_df["auc"],
            label=model, color=color,
            linewidth=1.8, marker="o",
            markersize=3, alpha=0.9)
    ax.fill_between(
        m_df["round"], m_df["auc"], m_df["auc"].iloc[0],
        alpha=0.07, color=color
    )

ax.axhline(0.5, color="red", linestyle="--",
           linewidth=0.8, alpha=0.5, label="Random (0.5)")
ax.set_xlabel("Federation Round", fontsize=11)
ax.set_ylabel("ROC-AUC", fontsize=11)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(linestyle="--", alpha=0.3)
ax.set_xlim(1, 20)
ax.set_ylim(0.55, 0.70)

plt.tight_layout()
for ext in ["png", "pdf"]:
    plt.savefig(
        FIG_DIR / f"fig3_mlp_resnet_auc_across_rounds.{ext}",
        dpi=300, bbox_inches="tight"
    )
plt.show()
print("✓ Saved: fig3_mlp_resnet_auc_across_rounds")

# ── Final summary ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("CELL 31 COMPLETE")
print("=" * 55)
print(f"Models compared     : {len(delta_df)}")
print(f"Models where FL ≥ C : "
      f"{(delta_df['AUC_delta_FL_vs_C'] >= 0).sum()}")
print(f"Models where FL < C : "
      f"{(delta_df['AUC_delta_FL_vs_C'] < 0).sum()}")
print(f"Best FL model       : "
      f"{delta_df.iloc[0]['Model']} "
      f"(AUC={delta_df.iloc[0]['FL_AUC']:.4f})")
print(f"Figures saved       : 3")
print(f"  fig1_fl_centralized_auc_grouped_bar")
print(f"  fig2_fl_auc_delta_vs_centralized")
print(f"  fig3_mlp_resnet_auc_across_rounds")
print("\nReady for Cell 32 : Feature group index mapping.")


## 8. Consent feature groups


In [ ]:
# Cell 32: Feature Group Index Mapping
# Maps FEATURE_GROUPS from Cell 5 config to encoded column indices
# This mapping is used in all subsequent masking and consent experiments
# Saves mapping to disk : kernel restart safe
# Requires: feature_columns (Cell 17), FEATURE_GROUPS (Cell 5)

import json
import pandas as pd
import numpy as np

# ── Load feature columns : kernel restart safe ────────────────
with open(TABLE_DIR / "feature_columns.json", "r") as f:
    feature_columns = json.load(f)

print(f"Total encoded feature columns: {len(feature_columns)}")

# ── Map each feature group to encoded column indices ──────────
# FEATURE_GROUPS defined in Cell 5 contains original column names
# After one-hot encoding column names expand e.g.
# "race" becomes "race_AfricanAmerican", "race_Asian" etc.
# Matching is done by substring so original names map correctly
# to all their encoded variants

feature_group_indices  = {}
feature_group_colnames = {}
unmatched_features     = []

print("\nFeature Group → Encoded Column Mapping")
print("=" * 65)

for group, raw_features in FEATURE_GROUPS.items():
    indices  = []
    colnames = []

    for feat in raw_features:
        # Find all encoded columns that contain this feature name
        matches = [
            (i, c) for i, c in enumerate(feature_columns)
            if feat.lower() == c.lower()           # exact match first
            or c.lower().startswith(feat.lower() + "_")  # one-hot prefix
        ]
        if matches:
            for idx, col in matches:
                indices.append(idx)
                colnames.append(col)
        else:
            unmatched_features.append((group, feat))

    # Deduplicate preserving order
    seen = set()
    unique_indices  = []
    unique_colnames = []
    for i, c in zip(indices, colnames):
        if i not in seen:
            seen.add(i)
            unique_indices.append(i)
            unique_colnames.append(c)

    feature_group_indices[group]  = unique_indices
    feature_group_colnames[group] = unique_colnames

    print(f"\n  {group} ({len(unique_indices)} columns)")
    print(f"  Original features : {raw_features}")
    print(f"  Encoded columns   : {unique_colnames}")
# ── Add Specialty group for unmapped medical_specialty columns ─
specialty_cols = [c for c in feature_columns 
                  if c.startswith("medical_specialty_")]
specialty_indices = [feature_columns.index(c) for c in specialty_cols]

feature_group_indices["Specialty"]   = specialty_indices
feature_group_colnames["Specialty"]  = specialty_cols
FEATURE_GROUPS["Specialty"]          = ["medical_specialty"]

print(f"\nSpecialty group added: {len(specialty_cols)} columns")
print(f"  e.g. {specialty_cols[:4]}")

# ── Coverage report ───────────────────────────────────────────
all_mapped = [i for idxs in feature_group_indices.values() for i in idxs]
all_mapped_set = set(all_mapped)
unmapped_cols  = [
    (i, c) for i, c in enumerate(feature_columns)
    if i not in all_mapped_set
]

print("\n" + "=" * 65)
print("COVERAGE SUMMARY")
print("=" * 65)
print(f"Total encoded columns   : {len(feature_columns)}")
print(f"Columns mapped to groups: {len(all_mapped_set)}")
print(f"Columns not in any group: {len(unmapped_cols)}")

if unmapped_cols:
    print(f"\nUnmapped columns (not assigned to any group):")
    for i, c in unmapped_cols[:20]:
        print(f"  [{i:>3}] {c}")
    if len(unmapped_cols) > 20:
        print(f"  ... and {len(unmapped_cols)-20} more")

if unmatched_features:
    print(f"\nFeatures from FEATURE_GROUPS not found in encoded matrix:")
    for group, feat in unmatched_features:
        print(f"  {group}: '{feat}'")


# ── Group size summary table ──────────────────────────────────
print("\n" + "=" * 65)
print("GROUP SIZE SUMMARY")
print("=" * 65)
print(f"{'Group':<15} {'N Columns':>10} {'% of Features':>15}")
print("-" * 42)
for group, idxs in feature_group_indices.items():
    pct = len(idxs) / len(feature_columns) * 100
    print(f"{group:<15} {len(idxs):>10} {pct:>14.1f}%")

# ── Save mapping to disk ──────────────────────────────────────
mapping_output = {
    "feature_columns"      : feature_columns,
    "feature_group_indices": feature_group_indices,
    "feature_group_colnames": feature_group_colnames,
    "unmapped_column_indices": [i for i, _ in unmapped_cols],
    "unmapped_column_names"  : [c for _, c in unmapped_cols],
}

with open(TABLE_DIR / "feature_group_mapping.json", "w") as f:
    json.dump(mapping_output, f, indent=2)

# Save as readable CSV too
mapping_rows = []
for group, idxs in feature_group_indices.items():
    for idx, col in zip(idxs, feature_group_colnames[group]):
        mapping_rows.append({
            "group"     : group,
            "col_index" : idx,
            "col_name"  : col,
        })
mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(
    TABLE_DIR / "feature_group_mapping.csv",
    index=False
)

print(f"\n✓ Saved: feature_group_mapping.json")
print(f"✓ Saved: feature_group_mapping.csv")
print(f"\nReady for Cell 33 : Feature masking experiment.")


## 9. Feature-group masking (Experiment A)


In [ ]:
# Cell 33: Feature Masking Experiment : Experiment 3
# Simulates patient consent to share only partial feature groups
# For each group: zero-mask those columns in BOTH train and test
# Retrains FL from scratch with masked client data each run
# Measures AUC drop vs standard FL baseline per group per model
# 6 groups x 3 models = 18 FL runs
# Requires: feature_group_mapping.json (Cell 32)
# Requires: client_data, X_test_np_fl, y_test_np_fl
# Requires: run_fl_experiment, optimal_thresholds, fl_feature_names

import json
import numpy as np
import pandas as pd
import time
# ── Patch run_fl_experiment to avoid pd scoping conflict ─────
def run_fl_experiment(
    client_data, model_name, model_template,
    X_test_np, y_test_np, optimal_thresholds,
    num_rounds=NUM_ROUNDS, experiment_name="Standard FL",
    feature_names=None
):
    from sklearn.base import clone
    threshold  = optimal_thresholds.get(model_name, 0.5)
    round_logs = []

    all_X = np.vstack([d["X"] for d in client_data.values()])
    all_y = np.concatenate([d["y"] for d in client_data.values()])

    init_size = min(2000, len(all_y))
    init_idx  = np.random.choice(len(all_y), init_size, replace=False)

    global_model = clone(model_template)
    if model_name == "LightGBM" and feature_names is not None:
        X_init_df = pd.DataFrame(all_X[init_idx], columns=feature_names)
        global_model.fit(X_init_df, all_y[init_idx])
    else:
        global_model.fit(all_X[init_idx], all_y[init_idx])

    aggregated = global_model
    print(f"    Running {experiment_name} - {model_name}")

    for rnd in range(1, num_rounds + 1):
        local_models = []
        client_sizes = []

        for client_name, data in client_data.items():
            X_c = data["X"]
            y_c = data["y"]
            if model_name == "Logistic Regression" and rnd > 1:
                local_model = clone(aggregated)
                local_model.fit(X_c, y_c)
            else:
                local_model = local_train(
                    model_template, X_c, y_c,
                    model_name, feature_names
                )
            local_models.append(local_model)
            client_sizes.append(len(y_c))

        aggregated = fedavg_aggregate(
            global_model, local_models,
            client_sizes, model_name
        )
        if model_name == "Logistic Regression":
            global_model = aggregated

        metrics = evaluate_model_np(
            aggregated, X_test_np, y_test_np,
            threshold, model_name
        )
        round_logs.append({
            "round"     : rnd,
            "model"     : model_name,
            "experiment": experiment_name,
            "auc"       : round(metrics["auc"], 4),
            "f1"        : round(metrics["f1"],  4),
            "accuracy"  : round(metrics["accuracy"], 4)
        })
        if rnd % 10 == 0 or rnd == 1:
            print(f"      Round {rnd:02d} | "
                  f"AUC: {metrics['auc']:.4f} | "
                  f"F1: {metrics['f1']:.4f}")

    final_metrics = evaluate_model_np(
        aggregated, X_test_np, y_test_np,
        threshold, model_name
    )
    return {
        "round_logs"    : pd.DataFrame(round_logs),
        "final_model"   : aggregated,
        "final_probs"   : final_metrics["probs"],
        "final_auc"     : final_metrics["auc"],
        "final_f1"      : final_metrics["f1"],
        "final_accuracy": final_metrics["accuracy"]
    }
# ── Load mapping : kernel restart safe ───────────────────────
with open(TABLE_DIR / "feature_group_mapping.json", "r") as f:
    mapping = json.load(f)

feature_group_indices  = mapping["feature_group_indices"]
feature_group_colnames = mapping["feature_group_colnames"]
feature_columns        = mapping["feature_columns"]

# ── Load FL baseline AUCs for delta calculation ───────────────
_baseline    = pd.read_csv(TABLE_DIR / "standard_fl_baseline_results.csv")
baseline_aucs = dict(zip(_baseline["Model"], _baseline["AUC"]))
print("Baseline AUCs loaded:", baseline_aucs)

# ── Recreate test arrays : kernel restart safe ────────────────
X_test_np_fl = X_test_scaled.values.astype(np.float64)
y_test_np_fl = y_test.values.astype(np.float64)

# ── Models for consent experiments ───────────────────────────
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

consent_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=50, scale_pos_weight=pos_weight,
        random_state=RANDOM_STATE, eval_metric="auc", verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=50, class_weight="balanced",
        random_state=RANDOM_STATE, verbose=-1
    ),
}

# ── Masking helper ────────────────────────────────────────────
def mask_group(X_np, indices):
    """
    Zero out specified column indices in a numpy array.
    Returns a copy : original data is never modified.
    Zeroing is the standard approach for feature masking in FL
    consent experiments : it removes information without
    changing the feature matrix shape.
    """
    X_masked = X_np.copy()
    X_masked[:, indices] = 0.0
    return X_masked

def apply_mask_to_clients(client_data, indices):
    """
    Returns a new client_data dict with specified columns
    zeroed in every hospital's training data.
    Original client_data is never modified.
    """
    masked_clients = {}
    for hosp, data in client_data.items():
        masked_clients[hosp] = {
            "X": mask_group(data["X"], indices),
            "y": data["y"].copy()
        }
    return masked_clients

# ── Main masking experiment loop ──────────────────────────────
print("\n" + "=" * 65)
print("EXPERIMENT 3 : FEATURE MASKING")
print("=" * 65)
print(f"Groups   : {list(feature_group_indices.keys())}")
print(f"Models   : {list(consent_models.keys())}")
print(f"Total FL runs : "
      f"{len(feature_group_indices) * len(consent_models)}")
print("=" * 65)

masking_results = []
all_round_logs  = []

for group_name, indices in feature_group_indices.items():
    print(f"\nGroup: {group_name} "
          f"({len(indices)} columns masked)")
    print("-" * 50)

    # Apply mask to both client training data and test data
    masked_clients  = apply_mask_to_clients(client_data, indices)
    X_test_masked   = mask_group(X_test_np_fl, indices)

    for model_name, model_template in consent_models.items():
        start = time.time()

        result = run_fl_experiment(
            client_data       = masked_clients,
            model_name        = model_name,
            model_template    = model_template,
            X_test_np         = X_test_masked,
            y_test_np         = y_test_np_fl,
            optimal_thresholds= optimal_thresholds,
            num_rounds        = NUM_ROUNDS,
            experiment_name   = f"Masking_{group_name}",
            feature_names     = fl_feature_names
        )

        elapsed      = round(time.time() - start, 1)
        baseline_auc = baseline_aucs.get(model_name, 0.0)
        auc_drop     = round(result["final_auc"] - baseline_auc, 4)

        masking_results.append({
            "Group"          : group_name,
            "N_cols_masked"  : len(indices),
            "Pct_cols_masked": round(len(indices)/len(feature_columns)*100, 1),
            "Model"          : model_name,
            "AUC"            : round(result["final_auc"], 4),
            "F1"             : round(result["final_f1"],  4),
            "Baseline_AUC"   : baseline_auc,
            "AUC_drop"       : auc_drop,
        })

        # Store round logs
        result["round_logs"]["group"] = group_name
        all_round_logs.append(result["round_logs"])

        direction = "▼" if auc_drop < 0 else "▲"
        print(f"  {model_name:<25} AUC={result['final_auc']:.4f}  "
              f"Drop={auc_drop:+.4f} {direction}  ({elapsed}s)")

# ── Save results ──────────────────────────────────────────────
masking_df   = pd.DataFrame(masking_results)
round_logs_df = pd.concat(all_round_logs, ignore_index=True)

masking_df.to_csv(
    TABLE_DIR / "exp3_feature_masking_results.csv",
    index=False
)
round_logs_df.to_csv(
    TABLE_DIR / "exp3_feature_masking_round_logs.csv",
    index=False
)

# ── Print summary table ───────────────────────────────────────
print("\n" + "=" * 65)
print("EXPERIMENT 3 : FEATURE MASKING RESULTS SUMMARY")
print("=" * 65)
print(f"{'Group':<15} {'Cols':>5} {'Model':<25} "
      f"{'AUC':>8} {'Drop':>10}")
print("-" * 65)
for _, row in masking_df.iterrows():
    direction = "▼" if row["AUC_drop"] < 0 else "▲"
    print(f"{row['Group']:<15} {row['N_cols_masked']:>5} "
          f"{row['Model']:<25} {row['AUC']:>8.4f} "
          f"{row['AUC_drop']:>+10.4f} {direction}")

# Most impactful group per model
print("\nMost impactful group per model (largest AUC drop):")
print("-" * 50)
for model in consent_models.keys():
    m_df    = masking_df[masking_df["Model"] == model]
    worst   = m_df.loc[m_df["AUC_drop"].idxmin()]
    print(f"  {model:<25} → {worst['Group']:<15} "
          f"Drop={worst['AUC_drop']:+.4f}")

print(f"\n✓ Saved: exp3_feature_masking_results.csv")
print(f"✓ Saved: exp3_feature_masking_round_logs.csv")
print(f"\nReady for Cell 34 : Feature masking visualisation.")


## 10. Age-subgroup consent and fairness (Experiment B)


In [ ]:

# Cell 35a : Exp 3b Setup + Scenario 3b-A: Older (>65) withdraw Observation
# GDPR Article 9: sensitive health data withdrawal by highest-risk age group
# 3 FL runs (one per model)
# Requires: client_data, X_test_np_fl, y_test_np_fl, X_test_df (Cell 35 pre-checks)
# Requires: feature_group_indices, consent_models, optimal_thresholds, baseline_aucs

import numpy as np
import pandas as pd
import time
from sklearn.metrics import roc_auc_score

# ── Subgroup masks (reused across 35a/b/c) ───────────────────
age_group_arr = X_test_df['_age_group'].values
race_arr      = X_test_df['_race'].values

mask_older  = (age_group_arr == 'Older (>65)')
mask_middle = (age_group_arr == 'Middle (46-65)')
mask_young  = (age_group_arr == 'Young (≤45)')
mask_cauc   = (race_arr == 'Caucasian')
mask_hisp   = (race_arr == 'Hispanic')

y_test_np = y_test_np_fl

# ── Helper: per-subgroup AUC ──────────────────────────────────
def subgroup_auc(probs, y_true_np, mask_bool):
    idx   = np.where(mask_bool)[0]
    y_sub = y_true_np[idx]
    p_sub = probs[idx]
    if y_sub.sum() < 30:
        return None
    try:
        return round(roc_auc_score(y_sub, p_sub), 4)
    except:
        return None

# ── Masking helpers ───────────────────────────────────────────
def apply_subgroup_mask_to_clients(client_data, col_indices,
                                   subgroup_val, feature_columns):
    age_col_idx = feature_columns.index('age')
    masked_clients = {}
    for hosp, data in client_data.items():
        X_c = data['X'].copy()
        y_c = data['y'].copy()
        if subgroup_val == 'Older (>65)':
            rows = X_c[:, age_col_idx] > 65
        elif subgroup_val == 'Middle (46-65)':
            rows = (X_c[:, age_col_idx] > 45) & (X_c[:, age_col_idx] <= 65)
        X_c[np.ix_(rows, col_indices)] = 0.0
        masked_clients[hosp] = {'X': X_c, 'y': y_c}
    return masked_clients

def apply_subgroup_mask_to_test(X_test_np, col_indices, test_mask_bool):
    X_masked = X_test_np.copy()
    X_masked[np.ix_(test_mask_bool, col_indices)] = 0.0
    return X_masked

# ── Indices ───────────────────────────────────────────────────
obs_indices = feature_group_indices['Observation']

# ── Storage (accumulates across 35a, 35b, 35c) ───────────────
exp3b_results  = []
exp3b_fairness = []

# ── Scenario 3b-A ─────────────────────────────────────────────
print("=" * 65)
print("SCENARIO 3b-A: Older (>65) withdraw Observation")
print("=" * 65)

masked_clients = apply_subgroup_mask_to_clients(
    client_data, obs_indices, 'Older (>65)', feature_columns)
X_test_masked = apply_subgroup_mask_to_test(
    X_test_np_fl, obs_indices, mask_older)

for model_name, model_template in consent_models.items():
    start  = time.time()
    result = run_fl_experiment(
        client_data        = masked_clients,
        model_name         = model_name,
        model_template     = model_template,
        X_test_np          = X_test_masked,
        y_test_np          = y_test_np_fl,
        optimal_thresholds = optimal_thresholds,
        num_rounds         = NUM_ROUNDS,
        experiment_name    = 'Exp3b-A',
        feature_names      = fl_feature_names
    )
    elapsed      = round(time.time() - start, 1)
    baseline_auc = baseline_aucs[model_name]
    auc_drop     = round(result['final_auc'] - baseline_auc, 4)
    probs        = result['final_probs']

    exp3b_results.append({
        'Scenario'   : '3b-A',
        'Description': 'Older (>65) withdraw Observation',
        'Model'      : model_name,
        'AUC'        : round(result['final_auc'], 4),
        'F1'         : round(result['final_f1'],  4),
        'Baseline_AUC': baseline_auc,
        'AUC_drop'   : auc_drop,
        'Time_s'     : elapsed,
    })
    exp3b_fairness.append({
        'Scenario'      : '3b-A',
        'Model'         : model_name,
        'AUC_overall'   : round(result['final_auc'], 4),
        'AUC_older'     : subgroup_auc(probs, y_test_np, mask_older),
        'AUC_middle'    : subgroup_auc(probs, y_test_np, mask_middle),
        'AUC_young'     : subgroup_auc(probs, y_test_np, mask_young),
        'AUC_caucasian' : subgroup_auc(probs, y_test_np, mask_cauc),
        'AUC_hispanic'  : subgroup_auc(probs, y_test_np, mask_hisp),
    })

    d = '▼' if auc_drop < 0 else '▲'
    print(f"  {model_name:<25} AUC={result['final_auc']:.4f}  "
          f"Drop={auc_drop:+.4f} {d}  ({elapsed}s)")

print(f"\n3b-A complete. exp3b_results has {len(exp3b_results)} rows.")
print("Run Cell 35b next.")


In [ ]:
# Cell 35b : Scenario 3b-B: Middle-age (46-65) withdraw Observation
# Control scenario : same feature group, different age cohort
# Requires: all setup from Cell 35a (masks, helpers, exp3b_results, exp3b_fairness)

import time

print("=" * 65)
print("SCENARIO 3b-B: Middle (46-65) withdraw Observation")
print("=" * 65)

masked_clients = apply_subgroup_mask_to_clients(
    client_data, obs_indices, 'Middle (46-65)', feature_columns)
X_test_masked = apply_subgroup_mask_to_test(
    X_test_np_fl, obs_indices, mask_middle)

for model_name, model_template in consent_models.items():
    start  = time.time()
    result = run_fl_experiment(
        client_data        = masked_clients,
        model_name         = model_name,
        model_template     = model_template,
        X_test_np          = X_test_masked,
        y_test_np          = y_test_np_fl,
        optimal_thresholds = optimal_thresholds,
        num_rounds         = NUM_ROUNDS,
        experiment_name    = 'Exp3b-B',
        feature_names      = fl_feature_names
    )
    elapsed      = round(time.time() - start, 1)
    baseline_auc = baseline_aucs[model_name]
    auc_drop     = round(result['final_auc'] - baseline_auc, 4)
    probs        = result['final_probs']

    exp3b_results.append({
        'Scenario'    : '3b-B',
        'Description' : 'Middle (46-65) withdraw Observation',
        'Model'       : model_name,
        'AUC'         : round(result['final_auc'], 4),
        'F1'          : round(result['final_f1'],  4),
        'Baseline_AUC': baseline_auc,
        'AUC_drop'    : auc_drop,
        'Time_s'      : elapsed,
    })
    exp3b_fairness.append({
        'Scenario'     : '3b-B',
        'Model'        : model_name,
        'AUC_overall'  : round(result['final_auc'], 4),
        'AUC_older'    : subgroup_auc(probs, y_test_np, mask_older),
        'AUC_middle'   : subgroup_auc(probs, y_test_np, mask_middle),
        'AUC_young'    : subgroup_auc(probs, y_test_np, mask_young),
        'AUC_caucasian': subgroup_auc(probs, y_test_np, mask_cauc),
        'AUC_hispanic' : subgroup_auc(probs, y_test_np, mask_hisp),
    })

    d = '▼' if auc_drop < 0 else '▲'
    print(f"  {model_name:<25} AUC={result['final_auc']:.4f}  "
          f"Drop={auc_drop:+.4f} {d}  ({elapsed}s)")

print(f"\n3b-B complete. exp3b_results has {len(exp3b_results)} rows.")
print("Run Cell 35c next.")


In [ ]:

# Cell 35c : Scenario 3b-C: Older (>65) full opt-out (all feature groups)
# Maximum withdrawal scenario : all 168 columns zeroed for older patients
# Tests whether Observation dominates or other groups compound the effect
# Requires: all setup from Cell 35a

import time

print("=" * 65)
print("SCENARIO 3b-C: Older (>65) full opt-out (all groups)")
print("=" * 65)

all_indices = list(range(len(feature_columns)))

masked_clients = apply_subgroup_mask_to_clients(
    client_data, all_indices, 'Older (>65)', feature_columns)
X_test_masked = apply_subgroup_mask_to_test(
    X_test_np_fl, all_indices, mask_older)

for model_name, model_template in consent_models.items():
    start  = time.time()
    result = run_fl_experiment(
        client_data        = masked_clients,
        model_name         = model_name,
        model_template     = model_template,
        X_test_np          = X_test_masked,
        y_test_np          = y_test_np_fl,
        optimal_thresholds = optimal_thresholds,
        num_rounds         = NUM_ROUNDS,
        experiment_name    = 'Exp3b-C',
        feature_names      = fl_feature_names
    )
    elapsed      = round(time.time() - start, 1)
    baseline_auc = baseline_aucs[model_name]
    auc_drop     = round(result['final_auc'] - baseline_auc, 4)
    probs        = result['final_probs']

    exp3b_results.append({
        'Scenario'    : '3b-C',
        'Description' : 'Older (>65) full opt-out all groups',
        'Model'       : model_name,
        'AUC'         : round(result['final_auc'], 4),
        'F1'          : round(result['final_f1'],  4),
        'Baseline_AUC': baseline_auc,
        'AUC_drop'    : auc_drop,
        'Time_s'      : elapsed,
    })
    exp3b_fairness.append({
        'Scenario'     : '3b-C',
        'Model'        : model_name,
        'AUC_overall'  : round(result['final_auc'], 4),
        'AUC_older'    : subgroup_auc(probs, y_test_np, mask_older),
        'AUC_middle'   : subgroup_auc(probs, y_test_np, mask_middle),
        'AUC_young'    : subgroup_auc(probs, y_test_np, mask_young),
        'AUC_caucasian': subgroup_auc(probs, y_test_np, mask_cauc),
        'AUC_hispanic' : subgroup_auc(probs, y_test_np, mask_hisp),
    })

    d = '▼' if auc_drop < 0 else '▲'
    print(f"  {model_name:<25} AUC={result['final_auc']:.4f}  "
          f"Drop={auc_drop:+.4f} {d}  ({elapsed}s)")

# ── Save all 3b results ───────────────────────────────────────
results_df  = pd.DataFrame(exp3b_results)
fairness_df = pd.DataFrame(exp3b_fairness)

results_df.to_csv(
    TABLE_DIR / 'exp3b_subgroup_optout_results.csv', index=False)
fairness_df.to_csv(
    TABLE_DIR / 'exp3b_subgroup_fairness.csv', index=False)

# ── Full summary ──────────────────────────────────────────────
print("\n" + "=" * 70)
print("EXPERIMENT 3b : COMPLETE RESULTS SUMMARY")
print("=" * 70)
print(f"\n{'Scenario':<6} {'Description':<38} {'Model':<25} "
      f"{'AUC':>7} {'Drop':>9}")
print("-" * 90)
for _, row in results_df.iterrows():
    d = '▼' if row['AUC_drop'] < 0 else '▲'
    print(f"{row['Scenario']:<6} {row['Description']:<38} "
          f"{row['Model']:<25} {row['AUC']:>7.4f} "
          f"{row['AUC_drop']:>+9.4f} {d}")

print("\n" + "=" * 70)
print("FAIRNESS BREAKDOWN (AUC per subgroup)")
print("=" * 70)
print(f"\n{'Sc':<5} {'Model':<25} {'Overall':>8} {'Older':>8} "
      f"{'Middle':>8} {'Young':>8} {'Caucasian':>10} {'Hispanic*':>10}")
print("-" * 85)
for _, row in fairness_df.iterrows():
    def fmt(v):
        return f"{v:.4f}" if v is not None else "   N/A"
    print(f"{row['Scenario']:<5} {row['Model']:<25} "
          f"{fmt(row['AUC_overall']):>8} "
          f"{fmt(row['AUC_older']):>8} "
          f"{fmt(row['AUC_middle']):>8} "
          f"{fmt(row['AUC_young']):>8} "
          f"{fmt(row['AUC_caucasian']):>10} "
          f"{fmt(row['AUC_hispanic']):>10}")

print("\n* Hispanic n=404, ~50 positives : indicative only.")
print(f"\n✓ Saved: exp3b_subgroup_optout_results.csv")
print(f"✓ Saved: exp3b_subgroup_fairness.csv")
print(f"\nReady for Cell 36 : Experiment 3b visualisation.")


In [ ]:
# Cell 35d Fairness Disparity Summary Table
# Computes AUC disparity (Young − Older gap) per scenario per model
# Builds Article 22 fairness argument from existing results
# No FL runs needed : pure analysis on saved CSVs
# Requires: exp3b_subgroup_fairness.csv, exp3b_subgroup_optout_results.csv

import pandas as pd
import numpy as np

# ── Load ──────────────────────────────────────────────────────
fairness_df = pd.read_csv(TABLE_DIR / 'exp3b_subgroup_fairness.csv')
results_df  = pd.read_csv(TABLE_DIR / 'exp3b_subgroup_optout_results.csv')

MODELS    = ['Logistic Regression', 'XGBoost', 'LightGBM']
SCENARIOS = ['3b-A', '3b-B', '3b-C']

# ── FL baseline subgroup AUCs (no masking : from standard FL) ─
# These are the reference point before any consent withdrawal
# Taken from Cell 33 standard FL baseline run
baseline_subgroup = {
    'Logistic Regression': {'older': 0.6474, 'middle': 0.6474,
                             'young': 0.6474, 'caucasian': 0.6474},
    'XGBoost'            : {'older': 0.6545, 'middle': 0.6545,
                             'young': 0.6545, 'caucasian': 0.6545},
    'LightGBM'           : {'older': 0.6757, 'middle': 0.6757,
                             'young': 0.6757, 'caucasian': 0.6757},
}
# Note: baseline uses overall AUC as proxy since no subgroup
# breakdown was collected at baseline : conservative estimate

# ── Disparity calculations ────────────────────────────────────
disparity_rows = []

for sc in SCENARIOS:
    for model in MODELS:
        row = fairness_df[
            (fairness_df['Scenario'] == sc) &
            (fairness_df['Model']    == model)
        ].iloc[0]

        res_row = results_df[
            (results_df['Scenario'] == sc) &
            (results_df['Model']    == model)
        ].iloc[0]

        auc_older  = row['AUC_older']
        auc_middle = row['AUC_middle']
        auc_young  = row['AUC_young']
        auc_cauc   = row['AUC_caucasian']
        auc_hisp   = row['AUC_hispanic']

        # Key disparity metrics
        young_older_gap   = round(auc_young  - auc_older,  4)
        young_middle_gap  = round(auc_young  - auc_middle, 4)
        middle_older_gap  = round(auc_middle - auc_older,  4)
        cauc_older_gap    = round(auc_cauc   - auc_older,  4)

        # Max disparity across all subgroup pairs
        all_aucs    = [auc_older, auc_middle, auc_young, auc_cauc]
        max_dispar  = round(max(all_aucs) - min(all_aucs), 4)

        disparity_rows.append({
            'Scenario'        : sc,
            'Model'           : model,
            'AUC_overall'     : row['AUC_overall'],
            'AUC_drop'        : res_row['AUC_drop'],
            'AUC_older'       : auc_older,
            'AUC_middle'      : auc_middle,
            'AUC_young'       : auc_young,
            'AUC_caucasian'   : auc_cauc,
            'AUC_hispanic'    : auc_hisp,
            'Gap_Young_Older' : young_older_gap,
            'Gap_Young_Middle': young_middle_gap,
            'Gap_Middle_Older': middle_older_gap,
            'Gap_Cauc_Older'  : cauc_older_gap,
            'Max_Disparity'   : max_dispar,
        })

disparity_df = pd.DataFrame(disparity_rows)

# ── Save ──────────────────────────────────────────────────────
disparity_df.to_csv(
    TABLE_DIR / 'exp3b_fairness_disparity.csv', index=False)

# ── Print summary ─────────────────────────────────────────────
print("=" * 75)
print("EXPERIMENT 3b : FAIRNESS DISPARITY SUMMARY")
print("=" * 75)

print(f"\n{'Scenario':<6} {'Model':<25} {'Overall':>8} {'Drop':>8} "
      f"{'Young':>7} {'Middle':>7} {'Older':>7} "
      f"{'Y-O Gap':>8} {'Max Disp':>9}")
print("-" * 90)

for _, row in disparity_df.iterrows():
    print(f"{row['Scenario']:<6} {row['Model']:<25} "
          f"{row['AUC_overall']:>8.4f} "
          f"{row['AUC_drop']:>+8.4f} "
          f"{row['AUC_young']:>7.4f} "
          f"{row['AUC_middle']:>7.4f} "
          f"{row['AUC_older']:>7.4f} "
          f"{row['Gap_Young_Older']:>+8.4f} "
          f"{row['Max_Disparity']:>9.4f}")

# ── Disparity escalation by scenario ─────────────────────────
print("\n" + "=" * 75)
print("DISPARITY ESCALATION (Max gap across subgroups, averaged over models)")
print("=" * 75)
print(f"\n{'Scenario':<6} {'Description':<38} "
      f"{'Avg Max Disparity':>18} {'Avg Y-O Gap':>12}")
print("-" * 76)

scenario_desc = {
    '3b-A': 'Older (>65) withdraw Observation',
    '3b-B': 'Middle (46-65) withdraw Observation',
    '3b-C': 'Older (>65) full opt-out'
}
for sc in SCENARIOS:
    sc_df   = disparity_df[disparity_df['Scenario'] == sc]
    avg_max = sc_df['Max_Disparity'].mean()
    avg_yog = sc_df['Gap_Young_Older'].mean()
    print(f"{sc:<6} {scenario_desc[sc]:<38} "
          f"{avg_max:>18.4f} {avg_yog:>+12.4f}")

# ── Article 22 flag ───────────────────────────────────────────
print("\n" + "=" * 75)
print("ARTICLE 22 FLAGS - Disparity threshold > 0.10")
print("=" * 75)
flagged = disparity_df[disparity_df['Max_Disparity'] > 0.10]
if len(flagged) > 0:
    for _, row in flagged.iterrows():
        print(f"  ⚠  {row['Scenario']} | {row['Model']:<25} | "
              f"Max disparity = {row['Max_Disparity']:.4f} "
              f"(Young {row['AUC_young']:.4f} vs "
              f"Older {row['AUC_older']:.4f})")
else:
    print("  No scenarios exceed disparity threshold of 0.10")

print(f"\n✓ Saved: exp3b_fairness_disparity.csv")
print(f"\nReady for Cell 37 : Experiment 4: Progressive consent restriction.")


## 11. Progressive restriction (Experiment C)


In [ ]:
# Cell 37 : Experiment 4: Progressive Consent Restriction
# Simulates patient withdrawing one feature group at a time
# Order: least impactful first → most impactful last (from Exp 3)
# Cumulative masking: each step adds previous groups + new group
# GDPR Article 7: right to withdraw consent incrementally
# 6 steps × 3 models = 18 FL runs
# Requires: client_data, X_test_np_fl, y_test_np_fl
# Requires: feature_group_indices, consent_models, optimal_thresholds
# Requires: baseline_aucs, run_fl_experiment, mask_group

import numpy as np
import pandas as pd
import time
from sklearn.metrics import roc_auc_score

# ── Load Exp 3 results to derive order dynamically ────────────
exp3_df = pd.read_csv(TABLE_DIR / 'exp3_feature_masking_results.csv')

# Average drop per group across models, sort ascending (least → most harmful)
group_order = (
    exp3_df.groupby('Group')['AUC_drop']
    .mean()
    .sort_values(ascending=False)  # least negative = least harmful first
    .index.tolist()
)
print("=== CUMULATIVE RESTRICTION ORDER (least → most harmful) ===")
cumulative_cols = 0
for i, g in enumerate(group_order):
    n = len(feature_group_indices[g])
    cumulative_cols += n
    avg_drop = exp3_df.groupby('Group')['AUC_drop'].mean()[g]
    print(f"  Step {i+1}: +{g:<15} "
          f"({n:>3} cols, cumulative {cumulative_cols:>3}/{len(feature_columns)}) "
          f"Exp3 avg_drop={avg_drop:+.4f}")

# ── Subgroup masks (from Cell 35a setup) ──────────────────────
# Rebuild if kernel restarted
age_group_arr = X_test_df['_age_group'].values
race_arr      = X_test_df['_race'].values
mask_older    = (age_group_arr == 'Older (>65)')
mask_middle   = (age_group_arr == 'Middle (46-65)')
mask_young    = (age_group_arr == 'Young (≤45)')
mask_cauc     = (race_arr == 'Caucasian')
mask_hisp     = (race_arr == 'Hispanic')
y_test_np     = y_test_np_fl

def subgroup_auc(probs, y_true, mask_bool):
    idx   = np.where(mask_bool)[0]
    y_sub = y_true[idx]
    p_sub = probs[idx]
    if y_sub.sum() < 30:
        return None
    try:
        return round(roc_auc_score(y_sub, p_sub), 4)
    except:
        return None

# ── Storage ───────────────────────────────────────────────────
exp4_results  = []
exp4_fairness = []

# ── Step 0 : Baseline (no masking, from saved results) ────────
print("\n=== STEP 0: BASELINE (from standard FL results) ===")
baseline_df = pd.read_csv(
    TABLE_DIR / 'standard_fl_baseline_results.csv')

for model in consent_models.keys():
    row = baseline_df[baseline_df['Model'] == model]
    if len(row) == 0:
        continue
    auc = row['AUC'].values[0]
    f1  = row['F1'].values[0]
    exp4_results.append({
        'Step'            : 0,
        'Groups_masked'   : 'None',
        'N_cols_masked'   : 0,
        'Pct_cols_masked' : 0.0,
        'Model'           : model,
        'AUC'             : auc,
        'F1'              : f1,
        'Baseline_AUC'    : baseline_aucs[model],
        'AUC_drop'        : 0.0,
    })
    print(f"  {model:<25} AUC={auc:.4f} (baseline)")

# ── Main loop : Steps 1-6 ─────────────────────────────────────
print("\n" + "=" * 70)
print("EXPERIMENT 4 : PROGRESSIVE CONSENT RESTRICTION")
print("=" * 70)
print(f"Models : {list(consent_models.keys())}")
print(f"Steps  : {len(group_order)}")
print(f"Runs   : {len(group_order) * len(consent_models)}")
print("=" * 70)

cumulative_indices = []
groups_so_far      = []

for step, group in enumerate(group_order, start=1):
    # Add this group to cumulative mask
    cumulative_indices += feature_group_indices[group]
    groups_so_far.append(group)
    n_cols  = len(cumulative_indices)
    pct     = round(n_cols / 168 * 100, 1)
    label   = ' + '.join(groups_so_far)

    print(f"\n{'─'*70}")
    print(f"Step {step}: Mask {group} "
          f"(cumulative: {n_cols} cols / {pct}%)")
    print(f"  Groups masked so far: {groups_so_far}")
    print(f"{'─'*70}")

    # Apply cumulative mask to clients and test set
    masked_clients = {}
    for hosp, data in client_data.items():
        X_c = data['X'].copy()
        X_c[:, cumulative_indices] = 0.0
        masked_clients[hosp] = {'X': X_c, 'y': data['y'].copy()}

    X_test_masked = X_test_np_fl.copy()
    X_test_masked[:, cumulative_indices] = 0.0

    for model_name, model_template in consent_models.items():
        start  = time.time()
        result = run_fl_experiment(
            client_data        = masked_clients,
            model_name         = model_name,
            model_template     = model_template,
            X_test_np          = X_test_masked,
            y_test_np          = y_test_np_fl,
            optimal_thresholds = optimal_thresholds,
            num_rounds         = NUM_ROUNDS,
            experiment_name    = f"Exp4_Step{step}_{group}",
            feature_names      = fl_feature_names
        )
        elapsed      = round(time.time() - start, 1)
        baseline_auc = baseline_aucs[model_name]
        auc_drop     = round(result['final_auc'] - baseline_auc, 4)
        probs        = result['final_probs']

        exp4_results.append({
            'Step'           : step,
            'Group_added'    : group,
            'Groups_masked'  : label,
            'N_cols_masked'  : n_cols,
            'Pct_cols_masked': pct,
            'Model'          : model_name,
            'AUC'            : round(result['final_auc'], 4),
            'F1'             : round(result['final_f1'],  4),
            'Baseline_AUC'   : baseline_auc,
            'AUC_drop'       : auc_drop,
            'Time_s'         : elapsed,
        })

        exp4_fairness.append({
            'Step'          : step,
            'Group_added'   : group,
            'Model'         : model_name,
            'AUC_overall'   : round(result['final_auc'], 4),
            'AUC_older'     : subgroup_auc(probs, y_test_np, mask_older),
            'AUC_middle'    : subgroup_auc(probs, y_test_np, mask_middle),
            'AUC_young'     : subgroup_auc(probs, y_test_np, mask_young),
            'AUC_caucasian' : subgroup_auc(probs, y_test_np, mask_cauc),
            'AUC_hispanic'  : subgroup_auc(probs, y_test_np, mask_hisp),
            'Gap_Young_Older': round(
                (subgroup_auc(probs, y_test_np, mask_young) or 0) -
                (subgroup_auc(probs, y_test_np, mask_older) or 0), 4),
        })

        d = '▼' if auc_drop < 0 else '▲'
        print(f"  {model_name:<25} "
              f"AUC={result['final_auc']:.4f}  "
              f"Drop={auc_drop:+.4f} {d}  "
              f"({elapsed}s)")

# ── Save ──────────────────────────────────────────────────────
results_df  = pd.DataFrame(exp4_results)
fairness_df = pd.DataFrame(exp4_fairness)

results_df.to_csv(
    TABLE_DIR / 'exp4_progressive_restriction_results.csv',
    index=False)
fairness_df.to_csv(
    TABLE_DIR / 'exp4_progressive_restriction_fairness.csv',
    index=False)

# ── Summary table ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("EXPERIMENT 4 : RESULTS SUMMARY")
print("=" * 70)
print(f"\n{'Step':>5} {'Group added':<15} {'Cols':>5} "
      f"{'Model':<25} {'AUC':>7} {'Drop':>9}")
print("-" * 70)
for _, row in results_df.iterrows():
    d = '▼' if row['AUC_drop'] < 0 else ('▲' if row['AUC_drop'] > 0 else '-')
    print(f"{int(row['Step']):>5} "
          f"{str(row.get('Group_added', 'Baseline')):<15} "
          f"{int(row['N_cols_masked']):>5} "
          f"{row['Model']:<25} "
          f"{row['AUC']:>7.4f} "
          f"{row['AUC_drop']:>+9.4f} {d}")

print(f"\n✓ Saved: exp4_progressive_restriction_results.csv")
print(f"✓ Saved: exp4_progressive_restriction_fairness.csv")
print(f"\nReady for Cell 38 : Experiment 4 visualisation.")


## 12. Four-method feature-importance validation


In [ ]:
# Cell 38b : Feature Group Importance: 4 Independent Validation Methods
# Fixed: tree models use weighted ensemble soft voting
# Fully data-driven, no hardcoded values

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

import shap

# ── Universal predict_proba wrapper ──────────────────────────
def ensemble_predict_proba(model_obj, X_np, feature_cols=None):
    """
    Handles both:
    - sklearn model (LR) → direct predict_proba
    - tuple (models_list, weights) → weighted soft voting
    """
    if isinstance(model_obj, tuple):
        models_list, weights = model_obj
        probs = np.zeros(len(X_np))
        for m, w in zip(models_list, weights):
            try:
                p = m.predict_proba(X_np)[:, 1]
            except Exception:
                if feature_cols is not None:
                    X_df = pd.DataFrame(X_np, columns=feature_cols)
                    p = m.predict_proba(X_df)[:, 1]
                else:
                    raise
            probs += w * p
        return probs
    else:
        try:
            return model_obj.predict_proba(X_np)[:, 1]
        except Exception:
            if feature_cols is not None:
                X_df = pd.DataFrame(X_np, columns=feature_cols)
                return model_obj.predict_proba(X_df)[:, 1]
            raise

# ── Verify wrapper on full test set ──────────────────────────
print("=== VERIFYING PREDICTION WRAPPER ===")
expected = {'Logistic Regression': 0.6474,
            'XGBoost'            : 0.6545,
            'LightGBM'           : 0.6757}

for mname, model in fl_baseline_models_trained.items():
    probs = ensemble_predict_proba(
        model, X_test_np_fl, feature_columns)
    auc   = roc_auc_score(y_test_np_fl, probs)
    diff  = abs(auc - expected[mname])
    status = '✓' if diff < 0.001 else '⚠ CHECK'
    print(f"  {mname:<25} AUC={auc:.4f}  "
          f"expected={expected[mname]:.4f}  {status}")

lr_model   = fl_baseline_models_trained['Logistic Regression']
xgb_model  = fl_baseline_models_trained['XGBoost']
lgbm_model = fl_baseline_models_trained['LightGBM']

GROUPS  = list(feature_group_indices.keys())
results = {g: {} for g in GROUPS}

# ══════════════════════════════════════════════════════════════
# METHOD 1 : PERMUTATION IMPORTANCE
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("METHOD 1 : PERMUTATION IMPORTANCE (n_repeats=10)")
print("="*65)

rng = np.random.RandomState(42)

# Baseline AUCs
baseline_aucs_perm = {}
for mname, model in [('LR',   lr_model),
                      ('XGB',  xgb_model),
                      ('LGBM', lgbm_model)]:
    probs = ensemble_predict_proba(
        model, X_test_np_fl, feature_columns)
    baseline_aucs_perm[mname] = roc_auc_score(
        y_test_np_fl, probs)

perm_results = []
for group in GROUPS:
    indices = feature_group_indices[group]
    drops   = {'LR': [], 'XGB': [], 'LGBM': []}

    for rep in range(10):
        X_perm = X_test_np_fl.copy()
        shuffled_idx = rng.permutation(X_perm.shape[0])
        X_perm[:, indices] = X_perm[shuffled_idx][:, indices]

        for mname, model in [('LR',   lr_model),
                               ('XGB',  xgb_model),
                               ('LGBM', lgbm_model)]:
            probs = ensemble_predict_proba(
                model, X_perm, feature_columns)
            auc  = roc_auc_score(y_test_np_fl, probs)
            drops[mname].append(
                auc - baseline_aucs_perm[mname])

    avg_drop = np.mean([
        np.mean(drops['LR']),
        np.mean(drops['XGB']),
        np.mean(drops['LGBM'])
    ])
    results[group]['perm_avg'] = avg_drop
    perm_results.append({
        'Group'   : group,
        'LR_drop' : round(np.mean(drops['LR']),   4),
        'XGB_drop': round(np.mean(drops['XGB']),  4),
        'LGBM_drop':round(np.mean(drops['LGBM']), 4),
        'Avg_drop': round(avg_drop, 4),
        'LR_std'  : round(np.std(drops['LR']),    4),
        'XGB_std' : round(np.std(drops['XGB']),   4),
        'LGBM_std': round(np.std(drops['LGBM']),  4),
    })
    print(f"  {group:<15} "
          f"LR={np.mean(drops['LR']):+.4f}  "
          f"XGB={np.mean(drops['XGB']):+.4f}  "
          f"LGBM={np.mean(drops['LGBM']):+.4f}  "
          f"avg={avg_drop:+.4f}")

perm_df = pd.DataFrame(perm_results).sort_values('Avg_drop')

# ══════════════════════════════════════════════════════════════
# METHOD 2 : SHAP (tree models only, sample=1000)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("METHOD 2 : SHAP FEATURE GROUP IMPORTANCE")
print("="*65)

sample_idx  = np.random.RandomState(42).choice(
    X_test_np_fl.shape[0], 1000, replace=False)
X_sample    = X_test_np_fl[sample_idx]
X_sample_df = pd.DataFrame(X_sample, columns=feature_columns)

shap_results = []
for mname, model_obj, use_df in [
    ('XGB',  xgb_model,  False),
    ('LGBM', lgbm_model, True)
]:
    print(f"\n  Computing SHAP for {mname}...")
    try:
        # Use first client model for SHAP
        # (ensemble SHAP requires summing : first model is representative)
        models_list, weights = model_obj
        # Weighted average SHAP across client models
        shap_aggregate = None
        for m, w in zip(models_list, weights):
            explainer = shap.TreeExplainer(m)
            if use_df:
                sv = explainer.shap_values(X_sample_df)
            else:
                sv = explainer.shap_values(X_sample)
            # Handle list output (binary classification)
            if isinstance(sv, list):
                sv = sv[1]
            if sv.ndim == 3:
                sv = sv[:, :, 1]
            if shap_aggregate is None:
                shap_aggregate = w * np.abs(sv)
            else:
                shap_aggregate += w * np.abs(sv)

        mean_abs_shap = shap_aggregate.mean(axis=0)

        for group in GROUPS:
            indices    = feature_group_indices[group]
            group_shap = float(mean_abs_shap[indices].mean())
            results[group][f'{mname}_shap'] = group_shap
            shap_results.append({
                'Model'        : mname,
                'Group'        : group,
                'Mean_abs_SHAP': round(group_shap, 6)
            })
            print(f"    {group:<15} mean|SHAP|={group_shap:.6f}")

    except Exception as e:
        print(f"  SHAP failed for {mname}: {e}")
        for group in GROUPS:
            shap_results.append({
                'Model': mname, 'Group': group,
                'Mean_abs_SHAP': 0.0
            })

shap_df    = pd.DataFrame(shap_results)
shap_pivot = shap_df.pivot(
    index='Group', columns='Model',
    values='Mean_abs_SHAP').round(6)
shap_pivot['Avg_SHAP'] = shap_pivot.mean(axis=1)
shap_pivot = shap_pivot.sort_values(
    'Avg_SHAP', ascending=False)

print("\n  SHAP summary:")
print(shap_pivot.to_string())

# ══════════════════════════════════════════════════════════════
# METHOD 3 : PEARSON / SPEARMAN CORRELATION
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("METHOD 3 : CORRELATION WITH TARGET")
print("="*65)

corr_results = []
for group in GROUPS:
    indices    = feature_group_indices[group]
    pearson_r  = []
    spearman_r = []
    for idx in indices:
        feat = X_test_np_fl[:, idx]
        if feat.std() == 0:
            continue
        pr, _ = pearsonr(feat, y_test_np_fl)
        sr, _ = spearmanr(feat, y_test_np_fl)
        pearson_r.append(abs(pr))
        spearman_r.append(abs(sr))

    avg_p = np.mean(pearson_r)  if pearson_r  else 0.0
    avg_s = np.mean(spearman_r) if spearman_r else 0.0
    results[group]['pearson']  = avg_p
    results[group]['spearman'] = avg_s
    corr_results.append({
        'Group'           : group,
        'Avg_abs_Pearson' : round(avg_p, 6),
        'Avg_abs_Spearman': round(avg_s, 6),
        'Max_abs_Pearson' : round(max(pearson_r)  if pearson_r  else 0, 6),
        'Max_abs_Spearman': round(max(spearman_r) if spearman_r else 0, 6),
    })
    print(f"  {group:<15} "
          f"Pearson={avg_p:.6f}  "
          f"Spearman={avg_s:.6f}")

corr_df = pd.DataFrame(corr_results).sort_values(
    'Avg_abs_Pearson', ascending=False)

# ══════════════════════════════════════════════════════════════
# METHOD 4 : MUTUAL INFORMATION
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("METHOD 4 : MUTUAL INFORMATION WITH TARGET")
print("="*65)

mi_scores = mutual_info_classif(
    X_test_np_fl, y_test_np_fl,
    random_state=42, n_neighbors=5)

mi_results = []
for group in GROUPS:
    indices  = feature_group_indices[group]
    group_mi = float(mi_scores[indices].mean())
    max_mi   = float(mi_scores[indices].max())
    results[group]['mi'] = group_mi
    mi_results.append({
        'Group' : group,
        'Avg_MI': round(group_mi, 6),
        'Max_MI': round(max_mi,   6),
        'Sum_MI': round(float(mi_scores[indices].sum()), 6),
    })
    print(f"  {group:<15} "
          f"avg_MI={group_mi:.6f}  "
          f"max_MI={max_mi:.6f}  "
          f"sum_MI={mi_scores[indices].sum():.6f}")

mi_df = pd.DataFrame(mi_results).sort_values(
    'Avg_MI', ascending=False)

# ══════════════════════════════════════════════════════════════
# COMBINED RANKING TABLE
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("COMBINED RANKING : ALL 4 METHODS")
print("="*65)

combined = pd.DataFrame({
    'Group'      : GROUPS,
    'Perm_drop'  : [results[g].get('perm_avg', 0)  for g in GROUPS],
    'SHAP_avg'   : [shap_pivot.loc[g, 'Avg_SHAP']
                    if g in shap_pivot.index else 0
                    for g in GROUPS],
    'Pearson'    : [results[g].get('pearson',  0)  for g in GROUPS],
    'Spearman'   : [results[g].get('spearman', 0)  for g in GROUPS],
    'MI_avg'     : [results[g].get('mi',       0)  for g in GROUPS],
})

# Rank: for Perm_drop, most negative = most important (rank ascending)
# For all others, highest = most important (rank descending)
combined['Rank_Perm'] = combined['Perm_drop'].rank(
    ascending=True).astype(int)
for col in ['SHAP_avg', 'Pearson', 'Spearman', 'MI_avg']:
    combined[f'Rank_{col}'] = combined[col].rank(
        ascending=False).astype(int)

rank_cols = [c for c in combined.columns
             if c.startswith('Rank_')]
combined['Avg_rank'] = combined[rank_cols].mean(axis=1).round(2)
combined = combined.sort_values('Avg_rank')

print(f"\n{'Group':<15} {'Perm':>9} {'SHAP':>9} "
      f"{'Pearson':>9} {'Spearman':>9} {'MI':>9} "
      f"| {'R_Perm':>6} {'R_SHAP':>6} {'R_P':>4} "
      f"{'R_S':>4} {'R_MI':>5} {'AvgR':>6}")
print("-" * 90)
for _, row in combined.iterrows():
    print(f"{row['Group']:<15} "
          f"{row['Perm_drop']:>+9.4f} "
          f"{row['SHAP_avg']:>9.6f} "
          f"{row['Pearson']:>9.6f} "
          f"{row['Spearman']:>9.6f} "
          f"{row['MI_avg']:>9.6f} "
          f"| {int(row['Rank_Perm']):>6} "
          f"{int(row['Rank_SHAP_avg']):>6} "
          f"{int(row['Rank_Pearson']):>4} "
          f"{int(row['Rank_Spearman']):>4} "
          f"{int(row['Rank_MI_avg']):>5} "
          f"{row['Avg_rank']:>6.2f}")

# ── Save ──────────────────────────────────────────────────────
perm_df.to_csv(
    TABLE_DIR / 'exp4_permutation_importance.csv', index=False)
shap_pivot.to_csv(
    TABLE_DIR / 'exp4_shap_importance.csv')
corr_df.to_csv(
    TABLE_DIR / 'exp4_correlation.csv', index=False)
mi_df.to_csv(
    TABLE_DIR / 'exp4_mutual_information.csv', index=False)
combined.to_csv(
    TABLE_DIR / 'exp4_combined_ranking.csv', index=False)

print(f"\n✓ Saved: exp4_permutation_importance.csv")
print(f"✓ Saved: exp4_shap_importance.csv")
print(f"✓ Saved: exp4_correlation.csv")
print(f"✓ Saved: exp4_mutual_information.csv")
print(f"✓ Saved: exp4_combined_ranking.csv")
print(f"\nReady for Cell 38c : Combined ranking visualisation.")
